# Laguna XS.2 — Cross-Capability Writeability Replication
## v8 · publication-oriented replication

v6/v7 found that **causal necessity and adaptation plasticity can dissociate**,
and that target-gradient accessibility contains useful information about where
a sparse MoE is writable.

v8 tests whether that result generalizes beyond frontend/UI.

### Default capabilities

The supplied CSV contains five complete capability datasets:

- `frontend` — the original v5/v6/v7 dataset, preserved for comparability;
- `python`;
- `sql`;
- `systems`;
- `algorithms`.

By default v8 runs the **four new capabilities**. Set
`CAPABILITIES_TO_RUN` below to include `frontend` if a full pipeline
replication of the original capability is desired.

For each capability \(q\), v8 estimates:

\[
C_q(E) = \text{causal necessity}
\]

\[
R_q(E) = \text{supervised routing access}
\]

\[
G_q(E) = \text{gradient accessibility}
\]

\[
A_q(E) = \text{actual held-out adaptation}
\]

and tests whether:

\[
\rho(G_q,A_q) > \rho(R_q,A_q),\ \rho(C_q,A_q)
\]

### Leakage discipline

For every capability:

- **selection** constructs causal/routing/gradient selectors;
- **train** is the only adaptation data;
- the 50+50 held-out set is deterministically split into:
  - 25 target + 25 control **atlas validation**;
  - 25 target + 25 control **untouched final test**.
- validation adaptation is used only for population-level \(C/R/G \to A\)
  analysis;
- **no final selector is chosen from validation adaptation**;
- final selector confirmation and parameter-budget curves use the untouched
  final test.

### Causal baseline fairness

Unlike v7's panel-only causal selector, v8 performs a hierarchical causal
search for **every capability**:

1. all 39 sparse layers are causally ablated;
2. the top 8 layers are searched with four randomized expert partitions;
3. leaf experts are individually validated;
4. the best exact causal expert becomes the K=1 causal selector.

### Population analysis

For every capability, 48 experts are sampled independently of all selector
scores:

- one random expert from each of the 39 sparse layers;
- nine additional globally random expert positions.

These 48 experts — and only these 48 — are used for primary population
correlations. Score-selected sentinels are reported separately.

### Final selector comparison

Matched one-expert training, three training-order seeds:

- hierarchical causal;
- global supervised routing;
- global raw target-gradient;
- global target-minus-control gradient.

### Budget curves

Matched K={1,2,4,8} routed-expert budgets for:

- causal;
- routing;
- raw gradient;
- gradient-specific.

All expensive phases checkpoint incrementally and resume from CSV files.

## 1 — Install dependencies

In [ ]:
# Keep the CUDA-enabled PyTorch build supplied by the instance.
# v8 intentionally avoids scipy/scikit-learn.
%pip -q install -U \
  "transformers==5.14.1" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib

## 2 — Runtime tuning for AWS g7e.2xlarge

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["MALLOC_ARENA_MAX"] = "4"

os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "2"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:512"
)

print("Runtime configured for AWS g7e.2xlarge.")

## 3 — Canonical imports

In [ ]:
import os
import gc
import json
import time
import types
import shutil
import hashlib
import platform
import importlib.util

from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

print("Core imports: PASS")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)

## 4 — Dependency verification

In [ ]:
required_modules = [
    "torch",
    "transformers",
    "accelerate",
    "huggingface_hub",
    "safetensors",
    "numpy",
    "pandas",
    "psutil",
    "tqdm",
    "matplotlib",
]

missing_modules = [
    name
    for name in required_modules
    if importlib.util.find_spec(name) is None
]

if missing_modules:
    raise ModuleNotFoundError(
        "Missing required modules: "
        + ", ".join(missing_modules)
        + ". Re-run the install cell once, restart the kernel, then Run All."
    )

if importlib.util.find_spec("transformers.conversion_mapping") is None:
    raise ModuleNotFoundError(
        "transformers.conversion_mapping is unavailable. "
        "Re-run the first install cell, restart the kernel, then Run All."
    )

print("Dependency verification: PASS")

## 5 — Hardware and storage preflight

In [ ]:
import os
import shutil
import platform
from pathlib import Path

import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"This notebook expects exactly one GPU; found {torch.cuda.device_count()}."
    )

props = torch.cuda.get_device_properties(0)

print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary or larger).")

if (os.cpu_count() or 0) < 8:
    print("WARNING: fewer than 8 logical CPUs detected.")

if ram.total / 2**30 < 58:
    print("WARNING: less than a 64-GiB-class host detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen_devices = set()

for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen_devices:
            continue
        seen_devices.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

## 6 — Load and validate the v8 cross-capability CSV

Expected schema:

```text
example_id,capability,split,kind,source_capability,topic,difficulty,prompt,reference
```

Each capability must contain exactly:

```text
selection / target   50
selection / control  50
train     / target   50
heldout   / target   50
heldout   / control  50
```

Lookup order:

1. `LAGUNA_V8_CSV`
2. `/home/ec2-user/workspace/laguna_v8_cross_capability_experiment.csv`
3. `/workspace/laguna_v8_cross_capability_experiment.csv`
4. `/mnt/data/laguna_v8_cross_capability_experiment.csv`

In [ ]:
csv_candidates = []

if os.environ.get("LAGUNA_V8_CSV"):
    csv_candidates.append(
        Path(os.environ["LAGUNA_V8_CSV"]).expanduser()
    )

csv_candidates.extend([
    Path("/home/ec2-user/workspace/laguna_v8_cross_capability_experiment.csv"),
    Path("/workspace/laguna_v8_cross_capability_experiment.csv"),
    Path("/mnt/data/laguna_v8_cross_capability_experiment.csv"),
])

EXPERIMENT_CSV = next(
    (p.resolve() for p in csv_candidates if p.exists()),
    None,
)

if EXPERIMENT_CSV is None:
    raise FileNotFoundError(
        "laguna_v8_cross_capability_experiment.csv not found. "
        "Set LAGUNA_V8_CSV to its full path."
    )

experiment_df = pd.read_csv(EXPERIMENT_CSV)

required_cols = {
    "example_id",
    "capability",
    "split",
    "kind",
    "source_capability",
    "topic",
    "difficulty",
    "prompt",
    "reference",
}

missing_cols = required_cols - set(experiment_df.columns)

if missing_cols:
    raise ValueError(
        f"Experiment CSV missing columns: {sorted(missing_cols)}"
    )

for col in [
    "example_id",
    "capability",
    "split",
    "kind",
    "source_capability",
    "topic",
    "difficulty",
    "prompt",
    "reference",
]:
    experiment_df[col] = (
        experiment_df[col]
        .astype(str)
        .str.strip()
    )

experiment_df["capability"] = experiment_df["capability"].str.lower()
experiment_df["split"] = experiment_df["split"].str.lower()
experiment_df["kind"] = experiment_df["kind"].str.lower()
experiment_df["source_capability"] = experiment_df["source_capability"].str.lower()

EXPECTED_CAPABILITIES = [
    "frontend",
    "python",
    "sql",
    "systems",
    "algorithms",
]

if sorted(experiment_df["capability"].unique()) != sorted(EXPECTED_CAPABILITIES):
    raise RuntimeError(
        "Capability set mismatch. Found: "
        + repr(sorted(experiment_df["capability"].unique()))
    )

if experiment_df["example_id"].duplicated().any():
    dupes = experiment_df.loc[
        experiment_df["example_id"].duplicated(),
        "example_id",
    ].head(10).tolist()
    raise RuntimeError(f"Duplicate example IDs: {dupes}")

if (experiment_df["prompt"].str.len() == 0).any():
    raise RuntimeError("Blank prompt detected.")

if (experiment_df["reference"].str.len() == 0).any():
    raise RuntimeError("Blank reference detected.")

expected_counts = {
    ("selection", "target"): 50,
    ("selection", "control"): 50,
    ("train", "target"): 50,
    ("heldout", "target"): 50,
    ("heldout", "control"): 50,
}

for capability in EXPECTED_CAPABILITIES:
    cap = experiment_df[
        experiment_df["capability"] == capability
    ]

    for (split, kind), expected in expected_counts.items():
        actual = len(
            cap[
                (cap["split"] == split)
                & (cap["kind"] == kind)
            ]
        )

        if actual != expected:
            raise RuntimeError(
                f"{capability} {split}/{kind}: "
                f"{actual} != {expected}"
            )

    split_prompts = {
        split: set(
            cap[
                cap["split"] == split
            ]["prompt"]
        )
        for split in ["selection", "train", "heldout"]
    }

    if split_prompts["selection"] & split_prompts["train"]:
        raise RuntimeError(
            f"{capability}: selection/train leakage"
        )

    if split_prompts["selection"] & split_prompts["heldout"]:
        raise RuntimeError(
            f"{capability}: selection/heldout leakage"
        )

    if split_prompts["train"] & split_prompts["heldout"]:
        raise RuntimeError(
            f"{capability}: train/heldout leakage"
        )

    for split in ["selection", "heldout"]:
        target_prompts = set(
            cap[
                (cap["split"] == split)
                & (cap["kind"] == "target")
            ]["prompt"]
        )

        control_prompts = set(
            cap[
                (cap["split"] == split)
                & (cap["kind"] == "control")
            ]["prompt"]
        )

        if target_prompts & control_prompts:
            raise RuntimeError(
                f"{capability}: target/control overlap in {split}"
            )


# New-capability controls are deliberately balanced across the
# four other capability target banks: 12/12/13/13 per split.
for capability in ["python", "sql", "systems", "algorithms"]:
    for split in ["selection", "heldout"]:
        controls = experiment_df[
            (experiment_df["capability"] == capability)
            & (experiment_df["split"] == split)
            & (experiment_df["kind"] == "control")
        ]

        source_counts = (
            controls["source_capability"]
            .value_counts()
            .to_dict()
        )

        if len(source_counts) != 4:
            raise RuntimeError(
                f"{capability} {split}: expected four control sources; "
                f"found {source_counts}"
            )

        if sorted(source_counts.values()) != [12, 12, 13, 13]:
            raise RuntimeError(
                f"{capability} {split}: unbalanced control sources "
                f"{source_counts}"
            )

print("Experiment CSV:", EXPERIMENT_CSV)
display(
    experiment_df.groupby(
        ["capability", "split", "kind"]
    ).size().rename("count").reset_index()
)
print("Dataset validation: PASS")

## 7 — v8 experiment configuration

In [ ]:
# Default: replicate on four capabilities not used as the main v7 target.
CAPABILITIES_TO_RUN = [
    "python",
    "sql",
    "systems",
    "algorithms",
]

# Set this manually if you also want to rerun frontend through v8.
# CAPABILITIES_TO_RUN = [
#     "frontend", "python", "sql", "systems", "algorithms"
# ]

unknown = sorted(
    set(CAPABILITIES_TO_RUN)
    - set(EXPECTED_CAPABILITIES)
)

if unknown:
    raise ValueError(
        f"Unknown capabilities: {unknown}"
    )

CONTROL_PENALTY = 0.75

# Global screening.
GLOBAL_GRAD_PROBE_CASES = 8
GLOBAL_GRAD_CHUNK = 8

# Hierarchical causal search.
CAUSAL_TOP_LAYERS = 8
CAUSAL_SEARCH_SEEDS = [17, 53, 101, 211]
CAUSAL_INITIAL_GROUP_SIZE = 32
CAUSAL_BEAM_WIDTH = 4

# Population + selector sentinels.
POPULATION_N = 48
SELECTOR_TOP_N = 8
V7_ANCHORS = [
    (36, 229),  # frontend causal/read example
    (36, 95),   # frontend gradient-specific/write example
    (30, 229),  # frontend broad-gradient winner
    (29, 194),  # v6 routing winner
    (25, 168),  # v6 gradient winner
    (33, 1),    # v7 routing winner
]

# Precise FP32 gradient geometry.
PRECISE_GRAD_CASES = 12

# Population adaptation atlas.
ATLAS_LR = 1e-5
ATLAS_EPOCHS = 1
ATLAS_MAX_UPDATES = 7
ATLAS_GRAD_ACCUM = 8
ATLAS_ORDER_SEED = 101

# Final confirmation.
CONFIRM_LR = 1e-5
CONFIRM_EPOCHS = 3
CONFIRM_MAX_UPDATES = 50
CONFIRM_GRAD_ACCUM = 8
CONFIRM_ORDER_SEEDS = [11, 23, 47]

# Parameter-budget curve.
RUN_BUDGET_CURVES = True
BUDGET_KS = [1, 2, 4, 8]
BUDGET_SEED = 11

# Deterministic split/panel seeds vary by capability.
BASE_SPLIT_SEED = 8200
BASE_PANEL_SEED = 6200

RESULTS_ROOT = WORK_ROOT / "laguna_xs2_v8_cross_capability"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Capabilities:", CAPABILITIES_TO_RUN)
print("Results root:", RESULTS_ROOT)

## 8 — Resolve official BF16 Laguna XS.2 checkpoint

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)

    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB."
        )

    print("Downloading Laguna XS.2 BF16...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors",
            "*.json",
            "*.py",
            "*.jinja",
            "LICENSE*",
            "README*",
        ],
        max_workers=2,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]

if missing:
    raise RuntimeError(f"Incomplete checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print("Checkpoint verification: PASS")

## 9 — Register Laguna checkpoint conversion mapping

In [ ]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "Transformers does not expose the native Laguna checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

## 10 — Load BF16 model directly onto the RTX PRO 6000

In [ ]:
import gc
import time
import torch
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(6, os.cpu_count() or 6))

try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if (
        ".mlp.experts" in k
        or ".mlp.gate" in k
        or "e_score_correction_bias" in k
    )
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")
print("BF16 load: PASS")

## 11 — Validate Laguna MoE architecture

In [ ]:
cfg = model.config

SPARSE_LAYERS = []

for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)

    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample_mlp = model.model.layers[SPARSE_LAYERS[0]].mlp

assert tuple(sample_mlp.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample_mlp.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample_mlp.experts.gate_up_proj[0].numel()
    + sample_mlp.experts.down_proj[0].numel()
)

print("Sparse layers:", SPARSE_LAYERS)
print(f"Params / expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("Architecture validation: PASS")

## 12 — Correct Laguna teacher-forcing format

In [ ]:
def chat_prefix_text(prompt):
    messages = [{"role": "user", "content": prompt}]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    start = 0

    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(
            f"Could not identify answer boundary for: {prompt!r}"
        )

    return full_ids, start

## 13 — Generic aligned scoring

In [ ]:
def build_scoring_batch(df):
    df = df.reset_index(drop=True).copy()

    parsed = [
        parse_case(r.prompt, r.reference)
        for r in df.itertuples(index=False)
    ]

    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, seq_len),
        dtype=torch.long,
    )

    targets = torch.full(
        (batch_size, max_ref),
        -100,
        dtype=torch.long,
    )

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1

        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "df": df,
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

@torch.inference_mode()
def score_batch(batch):
    import torch.nn.functional as F

    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    per_example = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example

    return result

# Core intervention, routing, gradient and surgery helpers

In [ ]:
from contextlib import contextmanager
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)

    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")

    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(
    layer_idx,
    expert_ids=None,
    zero_all_routed=False,
    renormalize=False,
):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward

    expert_ids = (
        []
        if expert_ids is None
        else [int(x) for x in expert_ids]
    )

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = (
            original_forward(hidden_states)
        )

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )

            keep = ~torch.isin(selected_experts, ids)

            routing_weights = (
                routing_weights
                * keep.to(routing_weights.dtype)
            )

            if renormalize:
                denom = routing_weights.sum(
                    dim=-1,
                    keepdim=True,
                )

                routing_weights = torch.where(
                    denom > 0,
                    routing_weights
                    / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return (
            router_logits,
            routing_weights,
            selected_experts,
        )

    gate.forward = types.MethodType(
        patched_forward,
        gate,
    )

    try:
        yield
    finally:
        gate.forward = original_forward

In [ ]:
def summarize_selection_delta(ablated_nll):
    delta = (
        np.asarray(ablated_nll)
        - SELECTION_BASE_NLL
    )

    target_delta = float(
        delta[selection_target_mask].mean()
    )

    control_delta = float(
        delta[selection_control_mask].mean()
    )

    causal_specificity = (
        target_delta
        - CONTROL_PENALTY
        * max(control_delta, 0.0)
    )

    return {
        "target_delta_nll": target_delta,
        "control_delta_nll": control_delta,
        "causal_specificity": causal_specificity,
        "per_example_delta": delta,
    }

def intervention_score(
    layer_idx,
    expert_ids,
    renormalize=False,
):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_batch(SELECTION_BATCH)

    return summarize_selection_delta(nll)

def bootstrap_specificity(
    per_example_delta,
    n_boot=5000,
    seed=123,
):
    rng = np.random.default_rng(seed)
    d = np.asarray(
        per_example_delta,
        dtype=np.float64,
    )

    target = d[selection_target_mask]
    control = d[selection_control_mask]

    vals = np.empty(
        int(n_boot),
        dtype=np.float64,
    )

    for i in range(int(n_boot)):
        t = rng.choice(
            target,
            size=len(target),
            replace=True,
        ).mean()

        c = rng.choice(
            control,
            size=len(control),
            replace=True,
        ).mean()

        vals[i] = (
            t
            - CONTROL_PENALTY
            * max(c, 0.0)
        )

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

In [ ]:
def make_training_case(
    prompt,
    reference,
    max_length=1024,
):
    prefix_text = chat_prefix_text(
        prompt
    )

    prefix_ids = tokenizer.encode(
        prefix_text,
        add_special_tokens=False,
    )

    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    if len(full_ids) > max_length:
        raise ValueError(
            f"{len(full_ids)} tokens > {max_length}"
        )

    start = 0

    for a, b in zip(
        prefix_ids,
        full_ids,
    ):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError(
            "Could not identify answer boundary."
        )

    input_ids = torch.tensor(
        full_ids,
        dtype=torch.long,
        device="cuda:0",
    ).unsqueeze(0)

    attention_mask = torch.ones_like(
        input_ids
    )

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        dtype=torch.long,
        device="cuda:0",
    )

    targets = torch.tensor(
        full_ids[start:],
        dtype=torch.long,
        device="cuda:0",
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

In [ ]:
def make_supervised_position_mask(batch):
    mask = torch.zeros_like(
        batch["attention_mask"],
        dtype=torch.bool,
    )

    valid_targets = batch["targets"].ne(-100)
    positions = batch["pred_positions"]

    for b in range(mask.shape[0]):
        valid = valid_targets[b]
        pos = positions[valid]
        mask[b, pos] = True

    return mask

@contextmanager
def capture_routing_with_mask(batch, token_mask):
    records = {}
    originals = []

    token_mask_flat = token_mask.reshape(-1).bool()

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(idx, original_forward):
            def patched(self, hidden_states):
                logits, weights, selected = original_forward(hidden_states)

                with torch.no_grad():
                    mask = token_mask_flat

                    if mask.numel() != selected.shape[0]:
                        raise RuntimeError(
                            f"Routing mask/token mismatch at layer {idx}: "
                            f"{mask.numel()} vs {selected.shape[0]}"
                        )

                    mask = mask.to(selected.device)

                    ids = selected[mask].reshape(-1).long()
                    ws = weights[mask].reshape(-1).float()

                    counts = torch.bincount(
                        ids,
                        minlength=cfg.num_experts,
                    )

                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )
                    wsum.scatter_add_(0, ids, ws)

                    records[int(idx)] = {
                        "tokens": int(mask.sum().item()),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return logits, weights, selected

            return patched

        gate.forward = types.MethodType(
            make_forward(layer_idx, original),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

In [ ]:
GLOBAL_GRAD_PROBE_CASES = 8
GLOBAL_GRAD_CHUNK = 8
RUN_GLOBAL_GRADIENT_SCAN = True

def _fused_expert_grad_norms(
    gate_up_grad,
    down_grad,
    chunk_size=8,
):
    if gate_up_grad is None or down_grad is None:
        raise RuntimeError(
            "Expected fused expert gradients, got None."
        )

    n = gate_up_grad.shape[0]
    out = torch.empty(
        n,
        dtype=torch.float64,
        device="cpu",
    )

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        gu = gate_up_grad[start:end].float()
        down = down_grad[start:end].float()

        sq = (
            gu.square().sum(dim=(1, 2))
            + down.square().sum(dim=(1, 2))
        )

        out[start:end] = (
            torch.sqrt(sq)
            .double()
            .cpu()
        )

        del gu, down, sq

    return out.numpy()

def _gradient_objective_for_cases(
    cases,
    max_cases,
):
    used = min(
        int(max_cases),
        len(cases),
    )

    if used <= 0:
        raise RuntimeError("No gradient probe cases.")

    for case in cases[:used]:
        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            out = model(
                input_ids=case["input_ids"],
                attention_mask=case["attention_mask"],
                use_cache=False,
                logits_to_keep=case["pred_positions"],
                return_dict=True,
            )

            logits = out.logits.float()

            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                case["targets"].reshape(-1),
            ) / used

        loss.backward()

        del out, logits, loss

def scan_layer_fused_gradients(
    layer_idx,
    target_cases,
    control_cases,
    max_cases=8,
):
    mlp = get_sparse_mlp(layer_idx)
    experts = mlp.experts

    gate_up = experts.gate_up_proj
    down = experts.down_proj

    if gate_up.dtype != torch.bfloat16 or down.dtype != torch.bfloat16:
        print(
            "WARNING: fused expert dtype is",
            gate_up.dtype,
            down.dtype,
        )

    for p in model.parameters():
        p.requires_grad_(False)

    gate_up.requires_grad_(True)
    down.requires_grad_(True)

    model.eval()

    try:
        gate_up.grad = None
        down.grad = None

        _gradient_objective_for_cases(
            target_cases,
            max_cases,
        )

        target_norm = _fused_expert_grad_norms(
            gate_up.grad,
            down.grad,
            GLOBAL_GRAD_CHUNK,
        )

        gate_up.grad = None
        down.grad = None
        torch.cuda.empty_cache()

        _gradient_objective_for_cases(
            control_cases,
            max_cases,
        )

        control_norm = _fused_expert_grad_norms(
            gate_up.grad,
            down.grad,
            GLOBAL_GRAD_CHUNK,
        )

        return target_norm, control_norm

    finally:
        gate_up.grad = None
        down.grad = None

        gate_up.requires_grad_(False)
        down.requires_grad_(False)

        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import types


class SurgicalExpertBank(nn.Module):
    """
    Exact-baseline delta surgery.

    Forward =
        original_frozen_expert_output
        + trainable_selected_expert_output
        - frozen_selected_expert_output

    At initialization:
        trainable_selected == frozen_selected

    therefore:
        delta == 0 exactly

    and the untouched model's original expert kernel remains the base path.
    """

    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(layer_idx), int(expert_id))
            for layer_idx, expert_id in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}

        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu_key = f"L{layer_idx}_E{expert_id}_gu"
            down_key = f"L{layer_idx}_E{expert_id}_down"

            # FP32 master parameters.
            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.params[down_key] = nn.Parameter(
                experts.down_proj[
                    expert_id
                ]
                .detach()
                .float()
                .clone()
            )

            self.key_map[
                (layer_idx, expert_id)
            ] = (
                gu_key,
                down_key,
            )

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(
            p.numel()
            for p in self.parameters()
        )

    def install(self):
        if self.installed:
            return

        bank = self

        selected_layers = sorted({
            layer_idx
            for layer_idx, _ in self.selected_pairs
        })

        for layer_idx in selected_layers:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            original_forward = experts.forward

            self.original_forwards[
                layer_idx
            ] = original_forward

            selected_ids = sorted({
                expert_id
                for l, expert_id in self.selected_pairs
                if l == layer_idx
            })

            def make_forward(
                idx,
                base_experts,
                original_fn,
                selected_expert_ids,
            ):
                def patched_forward(
                    self_experts,
                    hidden_states,
                    top_k_index,
                    top_k_weights,
                ):
                    # -------------------------------------------------
                    # IMPORTANT:
                    # Preserve Laguna's ORIGINAL execution path.
                    # -------------------------------------------------
                    base_output = original_fn(
                        hidden_states,
                        top_k_index,
                        top_k_weights,
                    )

                    correction = torch.zeros_like(
                        base_output
                    )

                    for expert_id in selected_expert_ids:

                        # top_k_index:
                        # [num_tokens, top_k]
                        token_idx, top_k_pos = torch.where(
                            top_k_index == expert_id
                        )

                        if token_idx.numel() == 0:
                            continue

                        current_state = hidden_states[
                            token_idx
                        ]

                        compute_dtype = (
                            current_state.dtype
                        )

                        gu_key, down_key = (
                            bank.key_map[
                                (idx, expert_id)
                            ]
                        )

                        # ---------------------------------------------
                        # Trainable FP32 master -> current compute dtype
                        # Gradient propagates through .to(dtype).
                        # ---------------------------------------------
                        train_gu = bank.params[
                            gu_key
                        ].to(compute_dtype)

                        train_down = bank.params[
                            down_key
                        ].to(compute_dtype)

                        # ---------------------------------------------
                        # Frozen reference expert.
                        # ---------------------------------------------
                        frozen_gu = (
                            base_experts
                            .gate_up_proj[
                                expert_id
                            ]
                            .detach()
                        )

                        frozen_down = (
                            base_experts
                            .down_proj[
                                expert_id
                            ]
                            .detach()
                        )

                        # ---------------------------------------------
                        # Trainable selected expert.
                        # ---------------------------------------------
                        train_gate, train_up = F.linear(
                            current_state,
                            train_gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        train_hidden = (
                            base_experts.act_fn(
                                train_gate
                            )
                            * train_up
                        )

                        train_hidden = F.linear(
                            train_hidden,
                            train_down,
                        )

                        # ---------------------------------------------
                        # Frozen selected expert using EXACT SAME
                        # manual computation as trainable branch.
                        #
                        # Therefore at initialization:
                        # train_hidden - frozen_hidden == 0.
                        # ---------------------------------------------
                        frozen_gate, frozen_up = F.linear(
                            current_state,
                            frozen_gu,
                        ).chunk(
                            2,
                            dim=-1,
                        )

                        frozen_hidden = (
                            base_experts.act_fn(
                                frozen_gate
                            )
                            * frozen_up
                        )

                        frozen_hidden = F.linear(
                            frozen_hidden,
                            frozen_down,
                        )

                        route_weight = top_k_weights[
                            token_idx,
                            top_k_pos,
                            None,
                        ]

                        train_hidden = (
                            train_hidden
                            * route_weight
                        )

                        frozen_hidden = (
                            frozen_hidden
                            * route_weight
                        )

                        delta = (
                            train_hidden
                            - frozen_hidden
                        ).to(
                            base_output.dtype
                        )

                        # Out-of-place index_add preserves autograd.
                        correction = correction.index_add(
                            0,
                            token_idx,
                            delta,
                        )

                    return (
                        base_output
                        + correction
                    )

                return patched_forward

            experts.forward = types.MethodType(
                make_forward(
                    layer_idx,
                    experts,
                    original_forward,
                    selected_ids,
                ),
                experts,
            )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        for layer_idx, original_forward in (
            self.original_forwards.items()
        ):
            get_sparse_mlp(
                layer_idx
            ).experts.forward = (
                original_forward
            )

        self.original_forwards.clear()
        self.installed = False

In [ ]:
def verify_routed_bank_equivalence(
    pair,
    probe_df=None,
    tol=1e-5,
):
    if probe_df is None:
        probe_df = selection_df.head(8)

    batch = build_scoring_batch(
        probe_df
    )

    before = score_batch(batch)

    bank = SurgicalExpertBank([pair])
    bank.install()

    try:
        after = score_batch(batch)
    finally:
        bank.restore()
        del bank, batch
        gc.collect()
        torch.cuda.empty_cache()

    diff = float(
        np.max(
            np.abs(before - after)
        )
    )

    print(
        "Routed bank equivalence",
        pair,
        "max ΔNLL:",
        diff,
    )

    if diff > tol:
        raise RuntimeError(
            "SurgicalExpertBank changes outputs before training."
        )

    return diff

In [ ]:
PRECISE_GRAD_CASES = 12
RUN_PRECISE_PANEL_GRADIENTS = True

def _configure_grad_checkpointing():
    if hasattr(
        model,
        "enable_input_require_grads",
    ):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            }
        )
    except TypeError:
        model.gradient_checkpointing_enable()

def _disable_grad_checkpointing():
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    if hasattr(
        model,
        "disable_input_require_grads",
    ):
        model.disable_input_require_grads()

def _backprop_bank_cases(
    bank,
    cases,
    max_cases,
):
    used = min(
        int(max_cases),
        len(cases),
    )

    for p in bank.parameters():
        p.grad = None

    for case in cases[:used]:
        with torch.autocast(
            "cuda",
            dtype=torch.bfloat16,
        ):
            out = model(
                input_ids=case["input_ids"],
                attention_mask=case["attention_mask"],
                use_cache=False,
                logits_to_keep=case["pred_positions"],
                return_dict=True,
            )

            logits = out.logits.float()

            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                case["targets"].reshape(-1),
            ) / used

        loss.backward()

        del out, logits, loss

    grads = []

    for p in bank.parameters():
        if p.grad is None:
            grads.append(
                torch.zeros_like(
                    p,
                    device="cpu",
                    dtype=torch.float32,
                )
            )
        else:
            grads.append(
                p.grad.detach().float().cpu().clone()
            )

    return grads

def precise_gradient_geometry(
    pair,
    target_cases,
    control_cases,
    max_cases=12,
):
    pair = tuple(map(int, pair))

    bank = SurgicalExpertBank([pair])
    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    _configure_grad_checkpointing()

    model.train()
    bank.train()

    try:
        target_grads = _backprop_bank_cases(
            bank,
            target_cases,
            max_cases,
        )

        control_grads = _backprop_bank_cases(
            bank,
            control_cases,
            max_cases,
        )

        target_sq = 0.0
        control_sq = 0.0
        dot = 0.0

        for gt, gc_ in zip(
            target_grads,
            control_grads,
        ):
            target_sq += float(
                torch.sum(gt * gt).item()
            )
            control_sq += float(
                torch.sum(gc_ * gc_).item()
            )
            dot += float(
                torch.sum(gt * gc_).item()
            )

        target_l2 = float(
            np.sqrt(target_sq)
        )
        control_l2 = float(
            np.sqrt(control_sq)
        )

        cosine = float(
            dot
            / max(
                target_l2 * control_l2,
                1e-12,
            )
        )

        return {
            "precise_target_grad_l2": target_l2,
            "precise_control_grad_l2": control_l2,
            "precise_grad_dot": dot,
            "precise_grad_cosine": cosine,
            "precise_grad_specific": (
                target_l2
                - CONTROL_PENALTY * control_l2
            ),
            "precise_grad_ratio": (
                target_l2
                / (control_l2 + 1e-12)
            ),
        }

    finally:
        bank.restore()
        _disable_grad_checkpointing()
        model.eval()

        del bank
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
@torch.inference_mode()
def score_batch_detailed(batch):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)

    nll = (
        (losses * valid).sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    preds = logits.argmax(dim=-1)

    correct = (
        preds.eq(targets)
        & valid
    )

    token_acc = (
        correct.sum(-1)
        / valid.sum(-1).clamp_min(1)
    )

    seq_exact = (
        (correct | ~valid)
        .all(dim=-1)
        .float()
    )

    result = {
        "nll": nll.cpu().numpy(),
        "token_acc": token_acc.cpu().numpy(),
        "seq_exact": seq_exact.cpu().numpy(),
    }

    del out, logits, losses, preds

    return result

def evaluate_batch_against_base(
    batch,
    df,
    base_detail,
):
    d = score_batch_detailed(batch)

    target_mask = (
        df["kind"].values == "target"
    )
    control_mask = (
        df["kind"].values == "control"
    )

    target_nll = float(
        d["nll"][target_mask].mean()
    )
    control_nll = float(
        d["nll"][control_mask].mean()
    )

    base_target_nll = float(
        base_detail["nll"][
            target_mask
        ].mean()
    )
    base_control_nll = float(
        base_detail["nll"][
            control_mask
        ].mean()
    )

    target_improvement = (
        base_target_nll - target_nll
    )

    control_improvement = (
        base_control_nll - control_nll
    )

    control_damage = (
        -control_improvement
    )

    utility_score = (
        target_improvement
        - CONTROL_PENALTY
        * max(control_damage, 0.0)
    )

    specific_gain = (
        target_improvement
        - control_improvement
    )

    return {
        "target_nll": target_nll,
        "control_nll": control_nll,
        "target_improvement": float(target_improvement),
        "control_improvement": float(control_improvement),
        "control_damage": float(control_damage),
        "utility_score": float(utility_score),
        "specific_gain": float(specific_gain),
        "target_token_acc": float(
            d["token_acc"][target_mask].mean()
        ),
        "control_token_acc": float(
            d["token_acc"][control_mask].mean()
        ),
        "target_seq_exact": float(
            d["seq_exact"][target_mask].mean()
        ),
        "control_seq_exact": float(
            d["seq_exact"][control_mask].mean()
        ),
        "per_example_nll": d["nll"],
    }

In [ ]:
def capture_routed_base_guard(pairs):
    guard = {}

    for layer_idx, expert_id in pairs:
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        guard[(layer_idx, expert_id)] = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu().clone(),
            experts.down_proj[
                expert_id
            ].detach().cpu().clone(),
        )

    return guard

def assert_routed_base_unchanged(guard):
    for (
        layer_idx,
        expert_id,
    ), (gu0, down0) in guard.items():
        experts = get_sparse_mlp(
            layer_idx
        ).experts

        gu1 = (
            experts.gate_up_proj[
                expert_id
            ].detach().cpu()
        )

        down1 = (
            experts.down_proj[
                expert_id
            ].detach().cpu()
        )

        if not torch.equal(gu0, gu1):
            raise RuntimeError(
                f"Frozen base gate_up changed: "
                f"L{layer_idx}/E{expert_id}"
            )

        if not torch.equal(down0, down1):
            raise RuntimeError(
                f"Frozen base down changed: "
                f"L{layer_idx}/E{expert_id}"
            )

def state_l2(state):
    total = 0.0

    for v in state.values():
        x = v.detach().float()
        total += float(
            torch.sum(x * x).item()
        )

    return float(np.sqrt(total))

def state_delta_l2(before, after):
    total = 0.0

    for k in before:
        d = (
            after[k].detach().float()
            - before[k].detach().float()
        )
        total += float(
            torch.sum(d * d).item()
        )

    return float(np.sqrt(total))

In [ ]:
def restore_routed_base_guard(guard):
    with torch.no_grad():
        for (
            layer_idx,
            expert_id,
        ), (gu0, down0) in guard.items():
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            experts.gate_up_proj[
                expert_id
            ].copy_(
                gu0.to(
                    device=experts.gate_up_proj.device,
                    dtype=experts.gate_up_proj.dtype,
                )
            )

            experts.down_proj[
                expert_id
            ].copy_(
                down0.to(
                    device=experts.down_proj.device,
                    dtype=experts.down_proj.dtype,
                )
            )

def copy_bank_into_base(
    bank,
    pairs,
):
    with torch.no_grad():
        for (
            layer_idx,
            expert_id,
        ) in pairs:
            experts = get_sparse_mlp(
                layer_idx
            ).experts

            gu_key, down_key = (
                bank.key_map[
                    (
                        layer_idx,
                        expert_id,
                    )
                ]
            )

            experts.gate_up_proj[
                expert_id
            ].copy_(
                bank.params[
                    gu_key
                ].detach().to(
                    device=experts.gate_up_proj.device,
                    dtype=experts.gate_up_proj.dtype,
                )
            )

            experts.down_proj[
                expert_id
            ].copy_(
                bank.params[
                    down_key
                ].detach().to(
                    device=experts.down_proj.device,
                    dtype=experts.down_proj.dtype,
                )
            )

def train_routed_experts(
    selector,
    pairs,
    order_seed,
    lr,
    epochs,
    max_updates,
    grad_accum,
    eval_batch,
    eval_df,
    eval_base_detail,
):
    """
    Train through exact-baseline delta surgery.

    Final reported metrics come from the physically merged BF16 expert slices
    evaluated through Laguna's native expert kernel, not from the delta path.
    The untouched BF16 base slices are restored before return.
    """
    pairs = sorted({
        tuple(map(int, pair))
        for pair in pairs
    })

    bank = SurgicalExpertBank(
        pairs
    )

    expected_params = (
        len(pairs)
        * params_per_expert
    )

    if (
        bank.trainable_parameter_count
        != expected_params
    ):
        raise RuntimeError(
            "Trainable parameter budget mismatch."
        )

    guard = capture_routed_base_guard(
        pairs
    )

    bank.install()

    for p in model.parameters():
        p.requires_grad_(False)

    _configure_grad_checkpointing()

    model.train()
    bank.train()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=float(lr),
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    initial_state = {
        k: v.detach().cpu().clone()
        for k, v in bank.state_dict().items()
    }

    rng = np.random.default_rng(
        int(order_seed)
    )

    history = []
    raw_step = 0
    update_step = 0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        for epoch in range(int(epochs)):
            order = rng.permutation(
                len(TRAIN_CASES)
            ).tolist()

            for position, case_idx in enumerate(
                order
            ):
                case = TRAIN_CASES[
                    int(case_idx)
                ]

                raw_step += 1

                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        input_ids=case[
                            "input_ids"
                        ],
                        attention_mask=case[
                            "attention_mask"
                        ],
                        use_cache=False,
                        logits_to_keep=case[
                            "pred_positions"
                        ],
                        return_dict=True,
                    )

                    logits = out.logits.float()

                    loss = F.cross_entropy(
                        logits.reshape(
                            -1,
                            logits.shape[-1],
                        ),
                        case[
                            "targets"
                        ].reshape(-1),
                    )

                    scaled = (
                        loss
                        / int(grad_accum)
                    )

                scaled.backward()

                is_final_available = (
                    epoch
                    == int(epochs) - 1
                    and position
                    == len(order) - 1
                )

                should_step = (
                    raw_step
                    % int(grad_accum)
                    == 0
                    or is_final_available
                )

                if should_step:
                    grad_norm = (
                        torch.nn.utils.clip_grad_norm_(
                            bank.parameters(),
                            1.0,
                        )
                    )

                    optimizer.step()

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    update_step += 1

                    history.append({
                        "selector": selector,
                        "budget_k": len(pairs),
                        "order_seed": int(order_seed),
                        "lr": float(lr),
                        "update_step": int(update_step),
                        "raw_step": int(raw_step),
                        "loss": float(
                            loss.detach().item()
                        ),
                        "grad_norm": float(
                            grad_norm
                        ),
                    })

                del out, logits, loss, scaled

                if (
                    update_step
                    >= int(max_updates)
                ):
                    break

            if (
                update_step
                >= int(max_updates)
            ):
                break

        model.eval()
        bank.eval()

        delta_metrics = (
            evaluate_batch_against_base(
                eval_batch,
                eval_df,
                eval_base_detail,
            )
        )

        final_state = {
            k: v.detach().cpu().clone()
            for k, v in bank.state_dict().items()
        }

        delta_l2 = state_delta_l2(
            initial_state,
            final_state,
        )

        base_l2 = state_l2(
            initial_state
        )

        # Return to Laguna's native expert forward before physical merge.
        bank.restore()

        try:
            copy_bank_into_base(
                bank,
                pairs,
            )

            model.eval()

            merged_metrics = (
                evaluate_batch_against_base(
                    eval_batch,
                    eval_df,
                    eval_base_detail,
                )
            )

            delta_merge_max_nll_diff = float(
                np.max(
                    np.abs(
                        np.asarray(
                            delta_metrics[
                                "per_example_nll"
                            ]
                        )
                        - np.asarray(
                            merged_metrics[
                                "per_example_nll"
                            ]
                        )
                    )
                )
            )

        finally:
            restore_routed_base_guard(
                guard
            )

        assert_routed_base_unchanged(
            guard
        )

        metrics = merged_metrics

        return {
            "selector": selector,
            "pairs": pairs,
            "budget_k": len(pairs),
            "order_seed": int(order_seed),
            "lr": float(lr),
            "epochs": int(epochs),
            "max_updates": int(max_updates),
            "updates": int(update_step),
            "trainable_params": int(
                bank.trainable_parameter_count
            ),
            "mean_grad_norm": float(
                np.mean([
                    h["grad_norm"]
                    for h in history
                ])
            ) if history else 0.0,
            "parameter_delta_l2": float(
                delta_l2
            ),
            "relative_parameter_delta": float(
                delta_l2
                / max(
                    base_l2,
                    1e-12,
                )
            ),
            "peak_gpu_gib": float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
            "delta_merge_max_nll_diff": (
                delta_merge_max_nll_diff
            ),
            **{
                k: v
                for k, v in metrics.items()
                if k
                != "per_example_nll"
            },
            "per_example_nll": metrics[
                "per_example_nll"
            ],
            "history": history,
        }

    finally:
        bank.restore()
        _disable_grad_checkpointing()
        model.eval()

        restore_routed_base_guard(
            guard
        )

        assert_routed_base_unchanged(
            guard
        )

        del optimizer
        del bank
        del guard

        gc.collect()
        torch.cuda.empty_cache()


# Dependency-free statistics helpers

In [ ]:
def _rankdata_average(x):
    """
    Average ranks for ties, implemented with pandas.
    """
    return (
        pd.Series(
            np.asarray(x, dtype=np.float64)
        )
        .rank(
            method="average",
            na_option="keep",
        )
        .to_numpy(dtype=np.float64)
    )

def _pearson_stat(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if (
        len(x) < 3
        or np.std(x) == 0
        or np.std(y) == 0
    ):
        return np.nan

    x = x - x.mean()
    y = y - y.mean()

    denom = np.sqrt(
        np.sum(x * x)
        * np.sum(y * y)
    )

    if denom <= 0:
        return np.nan

    return float(
        np.sum(x * y)
        / denom
    )

def _spearman_stat(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if len(x) < 3:
        return np.nan

    return _pearson_stat(
        _rankdata_average(x),
        _rankdata_average(y),
    )

def _kendall_tau_b(x, y):
    """
    Tie-aware Kendall tau-b.
    O(n^2), which is trivial for the 48-expert population panel.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    concordant = 0
    discordant = 0
    ties_x = 0
    ties_y = 0

    n = len(x)

    for i in range(n - 1):
        dx = x[i + 1:] - x[i]
        dy = y[i + 1:] - y[i]

        sx = np.sign(dx)
        sy = np.sign(dy)

        both_nonzero = (
            (sx != 0)
            & (sy != 0)
        )

        products = (
            sx[both_nonzero]
            * sy[both_nonzero]
        )

        concordant += int(
            np.sum(products > 0)
        )

        discordant += int(
            np.sum(products < 0)
        )

        ties_x += int(
            np.sum(
                (sx == 0)
                & (sy != 0)
            )
        )

        ties_y += int(
            np.sum(
                (sy == 0)
                & (sx != 0)
            )
        )

    denom = np.sqrt(
        (
            concordant
            + discordant
            + ties_x
        )
        * (
            concordant
            + discordant
            + ties_y
        )
    )

    if denom <= 0:
        return np.nan

    return float(
        (concordant - discordant)
        / denom
    )

def bootstrap_spearman(
    x,
    y,
    n_boot=5000,
    seed=1234,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    y = np.asarray(
        y,
        dtype=np.float64,
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(int(n_boot)):
        idx = rng.integers(
            0,
            len(x),
            size=len(x),
        )

        rho = _spearman_stat(
            x[idx],
            y[idx],
        )

        if np.isfinite(rho):
            vals.append(rho)

    vals = np.asarray(
        vals,
        dtype=np.float64,
    )

    if len(vals) == 0:
        return np.nan, np.nan

    return (
        float(
            np.quantile(
                vals,
                0.025,
            )
        ),
        float(
            np.quantile(
                vals,
                0.975,
            )
        ),
    )

def permutation_p_spearman(
    x,
    y,
    observed,
    n_perm=5000,
    seed=4321,
):
    """
    Two-sided permutation p-value for Spearman correlation.
    Avoids scipy and remains exact in spirit for the small panel.
    """
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    y = np.asarray(
        y,
        dtype=np.float64,
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    if not np.isfinite(observed):
        return np.nan

    rng = np.random.default_rng(seed)
    extreme = 0

    for _ in range(int(n_perm)):
        yp = rng.permutation(y)

        rho = _spearman_stat(
            x,
            yp,
        )

        if (
            np.isfinite(rho)
            and abs(rho)
            >= abs(observed)
        ):
            extreme += 1

    return float(
        (extreme + 1)
        / (int(n_perm) + 1)
    )

In [ ]:
def rank01(x):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    r = _rankdata_average(x)

    if len(r) <= 1:
        return np.zeros_like(r)

    return (
        (r - 1.0)
        / max(
            len(r) - 1.0,
            1.0,
        )
    )

def _linear_residuals(
    y,
    X,
):
    y = np.asarray(
        y,
        dtype=np.float64,
    )

    X = np.asarray(
        X,
        dtype=np.float64,
    )

    # Explicit intercept.
    X_design = np.column_stack([
        np.ones(
            len(X),
            dtype=np.float64,
        ),
        X,
    ])

    beta, *_ = np.linalg.lstsq(
        X_design,
        y,
        rcond=None,
    )

    fitted = (
        X_design
        @ beta
    )

    return (
        y
        - fitted
    )

def partial_rank_corr(
    df,
    x_col,
    y_col,
    controls,
):
    cols = [
        x_col,
        y_col,
        *controls,
    ]

    sub = (
        df[cols]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    if len(sub) < 5:
        return np.nan

    x = rank01(
        sub[x_col].values
    )

    y = rank01(
        sub[y_col].values
    )

    Z = np.column_stack([
        rank01(
            sub[c].values
        )
        for c in controls
    ])

    x_res = _linear_residuals(
        x,
        Z,
    )

    y_res = _linear_residuals(
        y,
        Z,
    )

    return _pearson_stat(
        x_res,
        y_res,
    )

# Capability context preparation

In [ ]:
CURRENT_CAPABILITY = None
RESULTS = None

def _capability_number(capability):
    return EXPECTED_CAPABILITIES.index(
        str(capability)
    )

def prepare_capability_context(capability):
    global CURRENT_CAPABILITY
    global RESULTS
    global selection_df
    global train_target_df
    global heldout_df
    global validation_df
    global final_test_df
    global SELECTION_BATCH
    global VALIDATION_BATCH
    global FINAL_TEST_BATCH
    global SELECTION_BASE_NLL
    global VALIDATION_BASE_NLL
    global FINAL_TEST_BASE_NLL
    global selection_target_mask
    global selection_control_mask
    global validation_target_mask
    global validation_control_mask
    global final_test_target_mask
    global final_test_control_mask
    global SELECTION_TARGET_CASES
    global SELECTION_CONTROL_CASES
    global TRAIN_CASES
    global VALIDATION_BASE_DETAIL
    global FINAL_TEST_BASE_DETAIL

    CURRENT_CAPABILITY = str(capability)

    cap_df = experiment_df[
        experiment_df["capability"]
        == CURRENT_CAPABILITY
    ].reset_index(drop=True)

    selection_df = cap_df[
        cap_df["split"] == "selection"
    ].reset_index(drop=True)

    train_target_df = cap_df[
        (cap_df["split"] == "train")
        & (cap_df["kind"] == "target")
    ].reset_index(drop=True)

    heldout_df = cap_df[
        cap_df["split"] == "heldout"
    ].reset_index(drop=True)

    split_seed = (
        BASE_SPLIT_SEED
        + 100 * _capability_number(
            CURRENT_CAPABILITY
        )
    )

    rng = np.random.default_rng(
        split_seed
    )

    validation_parts = []
    final_parts = []

    for kind in ["target", "control"]:
        part = heldout_df[
            heldout_df["kind"] == kind
        ].reset_index(drop=True)

        order = rng.permutation(
            len(part)
        )

        validation_parts.append(
            part.iloc[
                order[:25]
            ]
        )

        final_parts.append(
            part.iloc[
                order[25:]
            ]
        )

    validation_df = pd.concat(
        validation_parts,
        ignore_index=True,
    )

    final_test_df = pd.concat(
        final_parts,
        ignore_index=True,
    )

    assert len(selection_df) == 100
    assert len(train_target_df) == 50
    assert len(validation_df) == 50
    assert len(final_test_df) == 50
    assert int((validation_df["kind"] == "target").sum()) == 25
    assert int((validation_df["kind"] == "control").sum()) == 25
    assert int((final_test_df["kind"] == "target").sum()) == 25
    assert int((final_test_df["kind"] == "control").sum()) == 25

    validation_ids = set(
        validation_df["example_id"]
    )
    final_ids = set(
        final_test_df["example_id"]
    )

    if validation_ids & final_ids:
        raise RuntimeError(
            "Validation/final-test example leakage."
        )

    # Build GPU scoring batches.
    SELECTION_BATCH = build_scoring_batch(
        selection_df
    )
    VALIDATION_BATCH = build_scoring_batch(
        validation_df
    )
    FINAL_TEST_BATCH = build_scoring_batch(
        final_test_df
    )

    SELECTION_BASE_NLL = score_batch(
        SELECTION_BATCH
    )
    VALIDATION_BASE_NLL = score_batch(
        VALIDATION_BATCH
    )
    FINAL_TEST_BASE_NLL = score_batch(
        FINAL_TEST_BATCH
    )

    selection_target_mask = (
        selection_df["kind"].values
        == "target"
    )
    selection_control_mask = (
        selection_df["kind"].values
        == "control"
    )
    validation_target_mask = (
        validation_df["kind"].values
        == "target"
    )
    validation_control_mask = (
        validation_df["kind"].values
        == "control"
    )
    final_test_target_mask = (
        final_test_df["kind"].values
        == "target"
    )
    final_test_control_mask = (
        final_test_df["kind"].values
        == "control"
    )

    selection_target_df = selection_df[
        selection_df["kind"] == "target"
    ].reset_index(drop=True)

    selection_control_df = selection_df[
        selection_df["kind"] == "control"
    ].reset_index(drop=True)

    SELECTION_TARGET_CASES = [
        make_training_case(
            r.prompt,
            r.reference,
            1024,
        )
        for r in selection_target_df.itertuples(
            index=False
        )
    ]

    SELECTION_CONTROL_CASES = [
        make_training_case(
            r.prompt,
            r.reference,
            1024,
        )
        for r in selection_control_df.itertuples(
            index=False
        )
    ]

    TRAIN_CASES = [
        make_training_case(
            r.prompt,
            r.reference,
            1024,
        )
        for r in train_target_df.itertuples(
            index=False
        )
    ]

    VALIDATION_BASE_DETAIL = (
        score_batch_detailed(
            VALIDATION_BATCH
        )
    )

    FINAL_TEST_BASE_DETAIL = (
        score_batch_detailed(
            FINAL_TEST_BATCH
        )
    )

    RESULTS = (
        RESULTS_ROOT
        / CURRENT_CAPABILITY
    )
    RESULTS.mkdir(
        parents=True,
        exist_ok=True,
    )

    validation_df.to_csv(
        RESULTS / "atlas_validation_split.csv",
        index=False,
    )

    final_test_df.to_csv(
        RESULTS / "untouched_final_test_split.csv",
        index=False,
    )

    gc.collect()
    torch.cuda.empty_cache()

    print(
        f"\n=== {CURRENT_CAPABILITY.upper()} ==="
    )
    print("Results:", RESULTS)
    print(
        "Selection baseline target/control:",
        float(
            SELECTION_BASE_NLL[
                selection_target_mask
            ].mean()
        ),
        float(
            SELECTION_BASE_NLL[
                selection_control_mask
            ].mean()
        ),
    )

    return {
        "capability": CURRENT_CAPABILITY,
        "split_seed": split_seed,
        "results": str(RESULTS),
    }

## Scoring sanity check — run once before the long experiment

In [ ]:
prepare_capability_context(
    CAPABILITIES_TO_RUN[0]
)

demo_prefix = chat_prefix_text(
    selection_df.iloc[0]["prompt"]
)

print(
    "Teacher-forced boundary:",
    repr(
        (
            demo_prefix
            + "\n"
            + selection_df.iloc[0]["reference"]
        )[-100:]
    )
)

ids = SELECTION_BATCH["input_ids"][0:1]
mask = SELECTION_BATCH["attention_mask"][0:1]
pos = SELECTION_BATCH["position_ids"][0:1]
keep = SELECTION_BATCH["pred_positions"]

with torch.inference_mode():
    selective_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=keep,
        return_dict=True,
    ).logits.float()

    full_logits = model(
        input_ids=ids,
        attention_mask=mask,
        position_ids=pos,
        use_cache=False,
        logits_to_keep=0,
        return_dict=True,
    ).logits[:, keep, :].float()

max_diff = float(
    (
        selective_logits
        - full_logits
    ).abs().max().item()
)

print(
    "max |selective - full sliced logits|:",
    max_diff,
)

if max_diff > 1e-4:
    raise RuntimeError(
        "Selective-logit scorer does not match full-logit scoring."
    )

del selective_logits, full_logits
gc.collect()
torch.cuda.empty_cache()

verify_routed_bank_equivalence(
    (36, 229),
    tol=1e-5,
)

print("Global scoring/surgery sanity: PASS")

# Per-capability phase functions

In [ ]:
def run_global_routing_screen():
    path = RESULTS / "global_routing_screen.csv"

    if path.exists():
        df = pd.read_csv(path)

        expected = (
            len(SPARSE_LAYERS)
            * cfg.num_experts
        )

        if len(df) == expected:
            return df

    selection_target_df = selection_df[
        selection_df["kind"] == "target"
    ].reset_index(drop=True)

    batch = build_scoring_batch(
        selection_target_df
    )

    supervised_mask = (
        make_supervised_position_mask(
            batch
        )
    )

    with capture_routing_with_mask(
        batch,
        supervised_mask,
    ) as routing_records:
        _ = score_batch(batch)

    rows = []

    for layer_idx in SPARSE_LAYERS:
        rec = routing_records[
            int(layer_idx)
        ]

        tokens = max(
            1,
            rec["tokens"],
        )

        for expert_id in range(
            cfg.num_experts
        ):
            rows.append({
                "layer": int(layer_idx),
                "expert": int(expert_id),
                "selection_supervised_positions": int(
                    rec["tokens"]
                ),
                "selection_supervised_selected_hits": int(
                    rec["counts"][
                        expert_id
                    ].item()
                ),
                "selection_supervised_selected_rate": float(
                    rec["counts"][
                        expert_id
                    ].item()
                ) / tokens,
                "selection_supervised_routing_mass": float(
                    rec["weight_sums"][
                        expert_id
                    ].item()
                ) / tokens,
            })

    df = pd.DataFrame(
        rows
    ).sort_values(
        [
            "selection_supervised_routing_mass",
            "selection_supervised_selected_rate",
        ],
        ascending=False,
    ).reset_index(drop=True)

    df["routing_rank"] = (
        np.arange(
            len(df)
        )
        + 1
    )

    df.to_csv(
        path,
        index=False,
    )

    del batch
    gc.collect()
    torch.cuda.empty_cache()

    return df

In [ ]:
def run_global_gradient_screen():
    path = RESULTS / "global_gradient_screen.csv"

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed_layers = set()

    if not partial.empty:
        counts = partial.groupby(
            "layer"
        ).size()

        completed_layers = {
            int(layer)
            for layer, count in counts.items()
            if int(count) == cfg.num_experts
        }

    print(
        "Global gradient completed layers:",
        len(completed_layers),
        "/",
        len(SPARSE_LAYERS),
    )

    for layer_idx in tqdm(
        SPARSE_LAYERS,
        desc=f"{CURRENT_CAPABILITY}: global gradient",
    ):
        layer_idx = int(layer_idx)

        if layer_idx in completed_layers:
            continue

        torch.cuda.reset_peak_memory_stats()

        target_norm, control_norm = (
            scan_layer_fused_gradients(
                layer_idx,
                SELECTION_TARGET_CASES,
                SELECTION_CONTROL_CASES,
                max_cases=GLOBAL_GRAD_PROBE_CASES,
            )
        )

        layer_rows = []

        for expert_id in range(
            cfg.num_experts
        ):
            gt = float(
                target_norm[
                    expert_id
                ]
            )
            gc_ = float(
                control_norm[
                    expert_id
                ]
            )

            layer_rows.append({
                "layer": layer_idx,
                "expert": int(expert_id),
                "global_target_grad_bf16": gt,
                "global_control_grad_bf16": gc_,
                "global_grad_specific_bf16": (
                    gt
                    - CONTROL_PENALTY
                    * gc_
                ),
                "global_grad_ratio_bf16": (
                    gt
                    / (gc_ + 1e-12)
                ),
                "scan_peak_gpu_gib": float(
                    torch.cuda.max_memory_allocated()
                    / 2**30
                ),
            })

        if partial.empty:
            partial = pd.DataFrame(
                layer_rows
            )
        else:
            partial = pd.concat(
                [
                    partial[
                        partial["layer"]
                        != layer_idx
                    ],
                    pd.DataFrame(
                        layer_rows
                    ),
                ],
                ignore_index=True,
            )

        partial.to_csv(
            path,
            index=False,
        )

    df = pd.read_csv(path)

    expected = (
        len(SPARSE_LAYERS)
        * cfg.num_experts
    )

    if len(df) != expected:
        raise RuntimeError(
            f"Global gradient incomplete: "
            f"{len(df)} != {expected}"
        )

    return df

In [ ]:
def run_layer_causal_sweep():
    path = RESULTS / "layer_causal_scores.csv"

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed = set()

    if not partial.empty:
        completed = set(
            partial["layer"]
            .astype(int)
            .tolist()
        )

    for layer_idx in tqdm(
        SPARSE_LAYERS,
        desc=f"{CURRENT_CAPABILITY}: 39-layer causal sweep",
    ):
        layer_idx = int(layer_idx)

        if layer_idx in completed:
            continue

        with gate_intervention(
            layer_idx,
            zero_all_routed=True,
        ):
            nll = score_batch(
                SELECTION_BATCH
            )

        m = summarize_selection_delta(
            nll
        )

        row = {
            "layer": layer_idx,
            "target_delta_nll": m[
                "target_delta_nll"
            ],
            "control_delta_nll": m[
                "control_delta_nll"
            ],
            "causal_specificity": m[
                "causal_specificity"
            ],
        }

        partial = pd.concat(
            [
                partial[
                    partial["layer"]
                    != layer_idx
                ],
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        partial.to_csv(
            path,
            index=False,
        )

    df = pd.read_csv(path)

    if len(df) != len(SPARSE_LAYERS):
        raise RuntimeError(
            f"Layer causal sweep incomplete: "
            f"{len(df)} != {len(SPARSE_LAYERS)}"
        )

    return df.sort_values(
        "causal_specificity",
        ascending=False,
    ).reset_index(drop=True)

def hierarchical_expert_search(
    layer_idx,
    seed,
    initial_group_size=32,
    beam_width=4,
):
    rng = np.random.default_rng(
        int(seed)
    )

    order = rng.permutation(
        cfg.num_experts
    ).tolist()

    frontier = [
        order[
            i:i + initial_group_size
        ]
        for i in range(
            0,
            len(order),
            initial_group_size,
        )
    ]

    history = []
    level = 0

    while frontier:
        current = []

        for group in frontier:
            m = intervention_score(
                int(layer_idx),
                group,
            )

            rec = {
                "layer": int(layer_idx),
                "seed": int(seed),
                "level": int(level),
                "group_size": int(
                    len(group)
                ),
                "experts_json": json.dumps(
                    list(
                        map(
                            int,
                            group,
                        )
                    )
                ),
                "target_delta_nll": m[
                    "target_delta_nll"
                ],
                "control_delta_nll": m[
                    "control_delta_nll"
                ],
                "causal_specificity": m[
                    "causal_specificity"
                ],
            }

            history.append(
                rec
            )
            current.append(
                rec
            )

        current.sort(
            key=lambda x: x[
                "causal_specificity"
            ],
            reverse=True,
        )

        keep = current[
            :int(beam_width)
        ]

        if all(
            x["group_size"] == 1
            for x in keep
        ):
            break

        nxt = []

        for rec in keep:
            g = json.loads(
                rec[
                    "experts_json"
                ]
            )

            if len(g) == 1:
                nxt.append(g)
            else:
                mid = len(g) // 2
                nxt.extend([
                    g[:mid],
                    g[mid:],
                ])

        frontier = [
            x
            for x in nxt
            if x
        ]

        level += 1

    return pd.DataFrame(
        history
    )

def run_hierarchical_causal_search(
    layer_df,
):
    history_path = (
        RESULTS
        / "hierarchical_group_history.csv"
    )

    if history_path.exists():
        history_df = pd.read_csv(
            history_path
        )
    else:
        history_df = pd.DataFrame()

    candidate_layers = (
        layer_df.head(
            CAUSAL_TOP_LAYERS
        )["layer"]
        .astype(int)
        .tolist()
    )

    completed = set()

    if not history_df.empty:
        for (
            layer_idx,
            seed,
        ), g in history_df.groupby(
            ["layer", "seed"]
        ):
            if int(
                g["group_size"].min()
            ) == 1:
                completed.add(
                    (
                        int(layer_idx),
                        int(seed),
                    )
                )

    total_jobs = (
        len(candidate_layers)
        * len(
            CAUSAL_SEARCH_SEEDS
        )
    )

    print(
        "Hierarchical causal jobs complete:",
        len(completed),
        "/",
        total_jobs,
    )

    for layer_idx in candidate_layers:
        for seed in CAUSAL_SEARCH_SEEDS:
            key = (
                int(layer_idx),
                int(seed),
            )

            if key in completed:
                continue

            hist = hierarchical_expert_search(
                layer_idx,
                seed,
                initial_group_size=CAUSAL_INITIAL_GROUP_SIZE,
                beam_width=CAUSAL_BEAM_WIDTH,
            )

            if history_df.empty:
                history_df = hist
            else:
                history_df = pd.concat(
                    [
                        history_df[
                            ~(
                                (
                                    history_df["layer"]
                                    == int(layer_idx)
                                )
                                & (
                                    history_df["seed"]
                                    == int(seed)
                                )
                            )
                        ],
                        hist,
                    ],
                    ignore_index=True,
                )

            history_df.to_csv(
                history_path,
                index=False,
            )

    leaves = history_df[
        history_df[
            "group_size"
        ] == 1
    ].copy()

    leaf_pairs = sorted({
        (
            int(r.layer),
            int(
                json.loads(
                    r.experts_json
                )[0]
            ),
        )
        for r in leaves.itertuples(
            index=False
        )
    })

    if not leaf_pairs:
        raise RuntimeError(
            "Hierarchical causal search produced no leaf experts."
        )

    individual_path = (
        RESULTS
        / "hierarchical_individual_causal.csv"
    )

    if individual_path.exists():
        individual = pd.read_csv(
            individual_path
        )
    else:
        individual = pd.DataFrame()

    completed_pairs = set()

    if not individual.empty:
        completed_pairs = set(
            zip(
                individual[
                    "layer"
                ].astype(int),
                individual[
                    "expert"
                ].astype(int),
            )
        )

    for pair in tqdm(
        leaf_pairs,
        desc=f"{CURRENT_CAPABILITY}: exact causal leaves",
    ):
        if pair in completed_pairs:
            continue

        m = intervention_score(
            pair[0],
            [pair[1]],
            renormalize=False,
        )

        row = {
            "layer": pair[0],
            "expert": pair[1],
            "target_delta_nll": m[
                "target_delta_nll"
            ],
            "control_delta_nll": m[
                "control_delta_nll"
            ],
            "causal_specificity": m[
                "causal_specificity"
            ],
        }

        individual = pd.concat(
            [
                individual,
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        individual.to_csv(
            individual_path,
            index=False,
        )

    individual = pd.read_csv(
        individual_path
    ).sort_values(
        "causal_specificity",
        ascending=False,
    ).reset_index(drop=True)

    if len(individual) < 8:
        raise RuntimeError(
            "Too few exact causal candidates for K<=8 budget curves."
        )

    return history_df, individual

In [ ]:
def merge_global_screen(
    routing_df,
    gradient_df,
):
    df = gradient_df.merge(
        routing_df,
        on=[
            "layer",
            "expert",
        ],
        how="inner",
    )

    expected = (
        len(SPARSE_LAYERS)
        * cfg.num_experts
    )

    if len(df) != expected:
        raise RuntimeError(
            "Global screen merge mismatch."
        )

    df["global_target_grad_rank"] = (
        df[
            "global_target_grad_bf16"
        ]
        .rank(
            ascending=False,
            method="min",
        )
        .astype(int)
    )

    df[
        "global_grad_specific_rank"
    ] = (
        df[
            "global_grad_specific_bf16"
        ]
        .rank(
            ascending=False,
            method="min",
        )
        .astype(int)
    )

    df.to_csv(
        RESULTS / "global_screen_9984.csv",
        index=False,
    )

    return df

def build_atlas_panel(
    global_screen,
    causal_candidates,
):
    panel_seed = (
        BASE_PANEL_SEED
        + 100
        * _capability_number(
            CURRENT_CAPABILITY
        )
    )

    rng = np.random.default_rng(
        panel_seed
    )

    all_pairs = [
        (
            int(layer_idx),
            int(expert_id),
        )
        for layer_idx in SPARSE_LAYERS
        for expert_id in range(
            cfg.num_experts
        )
    ]

    population_pairs = []

    # One random expert per sparse layer.
    for layer_idx in SPARSE_LAYERS:
        expert_id = int(
            rng.integers(
                0,
                cfg.num_experts,
            )
        )

        population_pairs.append(
            (
                int(layer_idx),
                expert_id,
            )
        )

    population_set = set(
        population_pairs
    )

    remaining = [
        pair
        for pair in all_pairs
        if pair not in population_set
    ]

    extra_count = (
        POPULATION_N
        - len(
            population_pairs
        )
    )

    extra_idx = rng.choice(
        len(remaining),
        size=extra_count,
        replace=False,
    )

    for idx in extra_idx:
        population_pairs.append(
            remaining[
                int(idx)
            ]
        )

    population_set = set(
        population_pairs
    )

    if len(population_set) != POPULATION_N:
        raise RuntimeError(
            "Population panel size mismatch."
        )

    sentinel_candidates = []

    def append_ranked(
        df,
        sort_col,
        n,
    ):
        ranked = df.sort_values(
            sort_col,
            ascending=False,
        ).head(int(n))

        for r in ranked.itertuples(
            index=False
        ):
            sentinel_candidates.append(
                (
                    int(r.layer),
                    int(r.expert),
                )
            )

    append_ranked(
        global_screen,
        "global_target_grad_bf16",
        SELECTOR_TOP_N,
    )

    append_ranked(
        global_screen,
        "global_grad_specific_bf16",
        SELECTOR_TOP_N,
    )

    append_ranked(
        global_screen,
        "selection_supervised_routing_mass",
        SELECTOR_TOP_N,
    )

    append_ranked(
        causal_candidates,
        "causal_specificity",
        SELECTOR_TOP_N,
    )

    sentinel_candidates.extend(
        V7_ANCHORS
    )

    sentinel_pairs = []

    for pair in sentinel_candidates:
        pair = tuple(
            map(
                int,
                pair,
            )
        )

        if pair in population_set:
            continue

        if pair in sentinel_pairs:
            continue

        sentinel_pairs.append(
            pair
        )

    rows = []

    for pair in population_pairs:
        rows.append({
            "layer": pair[0],
            "expert": pair[1],
            "panel_group": "population_random",
        })

    for pair in sentinel_pairs:
        rows.append({
            "layer": pair[0],
            "expert": pair[1],
            "panel_group": "sentinel",
        })

    panel = pd.DataFrame(
        rows
    ).drop_duplicates(
        [
            "layer",
            "expert",
        ]
    ).reset_index(
        drop=True
    )

    pop_count = int(
        (
            panel["panel_group"]
            == "population_random"
        ).sum()
    )

    if pop_count != POPULATION_N:
        raise RuntimeError(
            f"Population panel count "
            f"{pop_count} != {POPULATION_N}"
        )

    panel = panel.merge(
        global_screen,
        on=[
            "layer",
            "expert",
        ],
        how="left",
    )

    panel.to_csv(
        RESULTS / "atlas_panel.csv",
        index=False,
    )

    return panel

In [ ]:
def run_panel_causal(panel):
    path = RESULTS / "panel_causal_scores.csv"

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed = set()

    if not partial.empty:
        completed = set(
            zip(
                partial["layer"].astype(int),
                partial["expert"].astype(int),
            )
        )

    for r in tqdm(
        list(
            panel.itertuples(
                index=False
            )
        ),
        desc=f"{CURRENT_CAPABILITY}: panel causal",
    ):
        pair = (
            int(r.layer),
            int(r.expert),
        )

        if pair in completed:
            continue

        normal = intervention_score(
            pair[0],
            [pair[1]],
            renormalize=False,
        )

        renorm = intervention_score(
            pair[0],
            [pair[1]],
            renormalize=True,
        )

        boot = bootstrap_specificity(
            normal[
                "per_example_delta"
            ],
            n_boot=5000,
            seed=(
                7000
                + pair[0] * 257
                + pair[1]
                + 100000
                * _capability_number(
                    CURRENT_CAPABILITY
                )
            ),
        )

        row = {
            "layer": pair[0],
            "expert": pair[1],
            "target_delta_nll": normal[
                "target_delta_nll"
            ],
            "control_delta_nll": normal[
                "control_delta_nll"
            ],
            "causal_specificity": normal[
                "causal_specificity"
            ],
            "renorm_target_delta_nll": renorm[
                "target_delta_nll"
            ],
            "renorm_control_delta_nll": renorm[
                "control_delta_nll"
            ],
            "renorm_causal_specificity": renorm[
                "causal_specificity"
            ],
            **boot,
        }

        partial = pd.concat(
            [
                partial,
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        partial.to_csv(
            path,
            index=False,
        )

    df = pd.read_csv(path)

    if len(df) != len(panel):
        raise RuntimeError(
            "Panel causal phase incomplete."
        )

    return df

In [ ]:
def run_panel_precise_gradients(panel):
    path = (
        RESULTS
        / "panel_precise_gradients.csv"
    )

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed = set()

    if not partial.empty:
        completed = set(
            zip(
                partial["layer"].astype(int),
                partial["expert"].astype(int),
            )
        )

    for r in tqdm(
        list(
            panel.itertuples(
                index=False
            )
        ),
        desc=f"{CURRENT_CAPABILITY}: precise gradients",
    ):
        pair = (
            int(r.layer),
            int(r.expert),
        )

        if pair in completed:
            continue

        metrics = precise_gradient_geometry(
            pair,
            SELECTION_TARGET_CASES,
            SELECTION_CONTROL_CASES,
            max_cases=PRECISE_GRAD_CASES,
        )

        partial = pd.concat(
            [
                partial,
                pd.DataFrame([
                    {
                        "layer": pair[0],
                        "expert": pair[1],
                        **metrics,
                    }
                ]),
            ],
            ignore_index=True,
        )

        partial.to_csv(
            path,
            index=False,
        )

    df = pd.read_csv(path)

    if len(df) != len(panel):
        raise RuntimeError(
            "Panel precise-gradient phase incomplete."
        )

    return df

In [ ]:
def run_panel_adaptation(panel):
    path = (
        RESULTS
        / "panel_adaptation_validation.csv"
    )

    nll_dir = (
        RESULTS
        / "atlas_validation_nll"
    )
    nll_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed = set()

    if not partial.empty:
        completed = set(
            zip(
                partial["layer"].astype(int),
                partial["expert"].astype(int),
            )
        )

    for r in tqdm(
        list(
            panel.itertuples(
                index=False
            )
        ),
        desc=f"{CURRENT_CAPABILITY}: adaptation atlas",
    ):
        pair = (
            int(r.layer),
            int(r.expert),
        )

        if pair in completed:
            continue

        result = train_routed_experts(
            selector=(
                f"{CURRENT_CAPABILITY}:"
                f"L{pair[0]}_E{pair[1]}"
            ),
            pairs=[pair],
            order_seed=ATLAS_ORDER_SEED,
            lr=ATLAS_LR,
            epochs=ATLAS_EPOCHS,
            max_updates=ATLAS_MAX_UPDATES,
            grad_accum=ATLAS_GRAD_ACCUM,
            eval_batch=VALIDATION_BATCH,
            eval_df=validation_df,
            eval_base_detail=VALIDATION_BASE_DETAIL,
        )

        row = {
            "layer": pair[0],
            "expert": pair[1],
            "atlas_order_seed": ATLAS_ORDER_SEED,
            "atlas_lr": ATLAS_LR,
            "atlas_updates": result[
                "updates"
            ],
            "trainable_params": result[
                "trainable_params"
            ],
            "mean_train_grad_norm": result[
                "mean_grad_norm"
            ],
            "parameter_delta_l2": result[
                "parameter_delta_l2"
            ],
            "relative_parameter_delta": result[
                "relative_parameter_delta"
            ],
            "peak_gpu_gib": result[
                "peak_gpu_gib"
            ],
            "delta_merge_max_nll_diff": result[
                "delta_merge_max_nll_diff"
            ],
            "validation_target_nll": result[
                "target_nll"
            ],
            "validation_control_nll": result[
                "control_nll"
            ],
            "validation_target_improvement": result[
                "target_improvement"
            ],
            "validation_control_improvement": result[
                "control_improvement"
            ],
            "validation_utility_score": result[
                "utility_score"
            ],
            "validation_specific_gain": result[
                "specific_gain"
            ],
            "validation_target_token_acc": result[
                "target_token_acc"
            ],
            "validation_control_token_acc": result[
                "control_token_acc"
            ],
            "validation_target_seq_exact": result[
                "target_seq_exact"
            ],
            "validation_control_seq_exact": result[
                "control_seq_exact"
            ],
        }

        partial = pd.concat(
            [
                partial,
                pd.DataFrame([row]),
            ],
            ignore_index=True,
        )

        partial.to_csv(
            path,
            index=False,
        )

        np.save(
            nll_dir
            / f"L{pair[0]}_E{pair[1]}.npy",
            result[
                "per_example_nll"
            ],
        )

        print(
            pair,
            "target=",
            f"{result['target_improvement']:+.4f}",
            "specific=",
            f"{result['specific_gain']:+.4f}",
        )

    df = pd.read_csv(path)

    if len(df) != len(panel):
        raise RuntimeError(
            "Panel adaptation phase incomplete."
        )

    return df

In [ ]:
def compute_population_correlations(
    atlas,
):
    pop = atlas[
        atlas["panel_group"]
        == "population_random"
    ].reset_index(drop=True)

    if len(pop) != POPULATION_N:
        raise RuntimeError(
            f"Population panel mismatch: "
            f"{len(pop)} != {POPULATION_N}"
        )

    predictors = {
        "causal_specificity": "causal_specificity",
        "renorm_causal_specificity": "renorm_causal_specificity",
        "routing_mass": "selection_supervised_routing_mass",
        "routing_rate": "selection_supervised_selected_rate",
        "global_target_gradient": "global_target_grad_bf16",
        "global_gradient_specific": "global_grad_specific_bf16",
        "precise_target_gradient": "precise_target_grad_l2",
        "precise_control_gradient": "precise_control_grad_l2",
        "precise_gradient_specific": "precise_grad_specific",
        "target_control_gradient_cosine": "precise_grad_cosine",
        "initial_to_train_gradient": "mean_train_grad_norm",
    }

    outcomes = {
        "target_improvement": "validation_target_improvement",
        "specific_gain": "validation_specific_gain",
    }

    rows = []

    for pred_name, pred_col in predictors.items():
        for out_name, out_col in outcomes.items():
            x = pop[
                pred_col
            ].to_numpy(
                dtype=np.float64
            )
            y = pop[
                out_col
            ].to_numpy(
                dtype=np.float64
            )

            valid = (
                np.isfinite(x)
                & np.isfinite(y)
            )

            x = x[valid]
            y = y[valid]

            rho = _spearman_stat(
                x,
                y,
            )

            tau = _kendall_tau_b(
                x,
                y,
            )

            ci_low, ci_high = (
                bootstrap_spearman(
                    x,
                    y,
                    n_boot=5000,
                    seed=(
                        17000
                        + 1000
                        * _capability_number(
                            CURRENT_CAPABILITY
                        )
                        + len(rows)
                    ),
                )
            )

            p_value = (
                permutation_p_spearman(
                    x,
                    y,
                    observed=rho,
                    n_perm=5000,
                    seed=(
                        27000
                        + 1000
                        * _capability_number(
                            CURRENT_CAPABILITY
                        )
                        + len(rows)
                    ),
                )
            )

            rows.append({
                "capability": CURRENT_CAPABILITY,
                "predictor": pred_name,
                "outcome": out_name,
                "n": int(len(x)),
                "spearman_rho": float(
                    rho
                ),
                "spearman_p": float(
                    p_value
                ),
                "spearman_ci_low": float(
                    ci_low
                ),
                "spearman_ci_high": float(
                    ci_high
                ),
                "kendall_tau": float(
                    tau
                ),
            })

    corr = pd.DataFrame(
        rows
    ).sort_values(
        [
            "outcome",
            "spearman_rho",
        ],
        ascending=[
            True,
            False,
        ],
    )

    partial_rows = []

    for pred in [
        "causal_specificity",
        "global_target_grad_bf16",
        "precise_target_grad_l2",
        "precise_grad_specific",
    ]:
        for outcome in [
            "validation_target_improvement",
            "validation_specific_gain",
        ]:
            rho = partial_rank_corr(
                pop,
                pred,
                outcome,
                controls=[
                    "selection_supervised_routing_mass",
                    "layer",
                ],
            )

            partial_rows.append({
                "capability": CURRENT_CAPABILITY,
                "predictor": pred,
                "outcome": outcome,
                "controls": (
                    "routing_mass + layer_depth"
                ),
                "partial_rank_corr": rho,
            })

    partial = pd.DataFrame(
        partial_rows
    )

    corr.to_csv(
        RESULTS / "population_correlations.csv",
        index=False,
    )

    partial.to_csv(
        RESULTS / "partial_rank_correlations.csv",
        index=False,
    )

    return corr, partial

In [ ]:
def choose_final_selectors(
    global_screen,
    causal_candidates,
    atlas,
):
    causal_pair = tuple(
        causal_candidates.iloc[0][
            ["layer", "expert"]
        ]
        .astype(int)
        .tolist()
    )

    routing_pair = tuple(
        global_screen.sort_values(
            "selection_supervised_routing_mass",
            ascending=False,
        )
        .iloc[0][
            ["layer", "expert"]
        ]
        .astype(int)
        .tolist()
    )

    raw_top = set(
        tuple(
            map(
                int,
                pair,
            )
        )
        for pair in (
            global_screen.sort_values(
                "global_target_grad_bf16",
                ascending=False,
            )
            .head(
                SELECTOR_TOP_N
            )[
                ["layer", "expert"]
            ]
            .to_numpy()
            .tolist()
        )
    )

    spec_top = set(
        tuple(
            map(
                int,
                pair,
            )
        )
        for pair in (
            global_screen.sort_values(
                "global_grad_specific_bf16",
                ascending=False,
            )
            .head(
                SELECTOR_TOP_N
            )[
                ["layer", "expert"]
            ]
            .to_numpy()
            .tolist()
        )
    )

    raw_candidates = atlas[
        atlas.apply(
            lambda r: (
                int(r["layer"]),
                int(r["expert"]),
            ) in raw_top,
            axis=1,
        )
    ]

    spec_candidates = atlas[
        atlas.apply(
            lambda r: (
                int(r["layer"]),
                int(r["expert"]),
            ) in spec_top,
            axis=1,
        )
    ]

    if len(raw_candidates) != SELECTOR_TOP_N:
        raise RuntimeError(
            "Not all top raw-gradient candidates are in the panel."
        )

    if len(spec_candidates) != SELECTOR_TOP_N:
        raise RuntimeError(
            "Not all top gradient-specific candidates are in the panel."
        )

    gradient_pair = tuple(
        raw_candidates.sort_values(
            "precise_target_grad_l2",
            ascending=False,
        )
        .iloc[0][
            ["layer", "expert"]
        ]
        .astype(int)
        .tolist()
    )

    gradient_specific_pair = tuple(
        spec_candidates.sort_values(
            "precise_grad_specific",
            ascending=False,
        )
        .iloc[0][
            ["layer", "expert"]
        ]
        .astype(int)
        .tolist()
    )

    selectors = {
        "causal": causal_pair,
        "routing": routing_pair,
        "gradient": gradient_pair,
        "gradient_specific": gradient_specific_pair,
    }

    for pair in sorted(
        set(
            selectors.values()
        )
    ):
        verify_routed_bank_equivalence(
            pair,
            tol=1e-5,
        )

    (
        RESULTS
        / "final_selectors.json"
    ).write_text(
        json.dumps(
            {
                k: list(
                    map(
                        int,
                        v,
                    )
                )
                for k, v in selectors.items()
            },
            indent=2,
        )
    )

    return selectors

In [ ]:
def run_final_confirmation(
    selectors,
):
    path = (
        RESULTS
        / "final_confirmation.csv"
    )

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed = set()

    if not partial.empty:
        completed = set(
            zip(
                partial[
                    "selector"
                ].astype(str),
                partial[
                    "order_seed"
                ].astype(int),
            )
        )

    for selector, pair in selectors.items():
        for seed in CONFIRM_ORDER_SEEDS:
            key = (
                str(selector),
                int(seed),
            )

            if key in completed:
                continue

            print(
                "\nFINAL TEST",
                CURRENT_CAPABILITY,
                selector,
                pair,
                "seed",
                seed,
            )

            result = train_routed_experts(
                selector=(
                    f"{CURRENT_CAPABILITY}:"
                    f"{selector}"
                ),
                pairs=[pair],
                order_seed=seed,
                lr=CONFIRM_LR,
                epochs=CONFIRM_EPOCHS,
                max_updates=CONFIRM_MAX_UPDATES,
                grad_accum=CONFIRM_GRAD_ACCUM,
                eval_batch=FINAL_TEST_BATCH,
                eval_df=final_test_df,
                eval_base_detail=FINAL_TEST_BASE_DETAIL,
            )

            row = {
                "capability": CURRENT_CAPABILITY,
                "selector": selector,
                "layer": pair[0],
                "expert": pair[1],
                "order_seed": int(seed),
                "lr": CONFIRM_LR,
                "updates": result[
                    "updates"
                ],
                "trainable_params": result[
                    "trainable_params"
                ],
                "mean_train_grad_norm": result[
                    "mean_grad_norm"
                ],
                "relative_parameter_delta": result[
                    "relative_parameter_delta"
                ],
                "final_target_improvement": result[
                    "target_improvement"
                ],
                "final_control_improvement": result[
                    "control_improvement"
                ],
                "final_utility_score": result[
                    "utility_score"
                ],
                "final_specific_gain": result[
                    "specific_gain"
                ],
                "final_target_token_acc": result[
                    "target_token_acc"
                ],
                "final_control_token_acc": result[
                    "control_token_acc"
                ],
                "final_target_seq_exact": result[
                    "target_seq_exact"
                ],
                "final_control_seq_exact": result[
                    "control_seq_exact"
                ],
                "peak_gpu_gib": result[
                    "peak_gpu_gib"
                ],
                "delta_merge_max_nll_diff": result[
                    "delta_merge_max_nll_diff"
                ],
            }

            partial = pd.concat(
                [
                    partial,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            partial.to_csv(
                path,
                index=False,
            )

    final_df = pd.read_csv(path)

    expected = (
        len(selectors)
        * len(
            CONFIRM_ORDER_SEEDS
        )
    )

    if len(final_df) != expected:
        raise RuntimeError(
            f"Final confirmation incomplete: "
            f"{len(final_df)} != {expected}"
        )

    summary = (
        final_df.groupby(
            [
                "capability",
                "selector",
                "layer",
                "expert",
            ]
        )
        .agg(
            n_runs=(
                "order_seed",
                "count",
            ),
            target_improvement_mean=(
                "final_target_improvement",
                "mean",
            ),
            target_improvement_std=(
                "final_target_improvement",
                "std",
            ),
            control_improvement_mean=(
                "final_control_improvement",
                "mean",
            ),
            specific_gain_mean=(
                "final_specific_gain",
                "mean",
            ),
            specific_gain_std=(
                "final_specific_gain",
                "std",
            ),
            target_token_acc_mean=(
                "final_target_token_acc",
                "mean",
            ),
            target_seq_exact_mean=(
                "final_target_seq_exact",
                "mean",
            ),
            mean_grad_norm=(
                "mean_train_grad_norm",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            "target_improvement_mean",
            ascending=False,
        )
    )

    summary.to_csv(
        RESULTS
        / "final_confirmation_summary.csv",
        index=False,
    )

    return final_df, summary

In [ ]:
def run_budget_curve(
    global_screen,
    causal_candidates,
):
    path = RESULTS / "budget_curve.csv"

    if not RUN_BUDGET_CURVES:
        return pd.DataFrame()

    gradient_order = [
        (
            int(r.layer),
            int(r.expert),
        )
        for r in global_screen.sort_values(
            "global_target_grad_bf16",
            ascending=False,
        ).itertuples(
            index=False
        )
    ]

    gradient_specific_order = [
        (
            int(r.layer),
            int(r.expert),
        )
        for r in global_screen.sort_values(
            "global_grad_specific_bf16",
            ascending=False,
        ).itertuples(
            index=False
        )
    ]

    routing_order = [
        (
            int(r.layer),
            int(r.expert),
        )
        for r in global_screen.sort_values(
            "selection_supervised_routing_mass",
            ascending=False,
        ).itertuples(
            index=False
        )
    ]

    causal_order = [
        (
            int(r.layer),
            int(r.expert),
        )
        for r in causal_candidates.sort_values(
            "causal_specificity",
            ascending=False,
        ).itertuples(
            index=False
        )
    ]

    methods = {
        "causal": causal_order,
        "routing": routing_order,
        "gradient": gradient_order,
        "gradient_specific": gradient_specific_order,
    }

    for method, order in methods.items():
        if len(order) < max(
            BUDGET_KS
        ):
            raise RuntimeError(
                f"{method} ranking has only "
                f"{len(order)} experts."
            )

    if path.exists():
        partial = pd.read_csv(path)
    else:
        partial = pd.DataFrame()

    completed = set()

    if not partial.empty:
        completed = set(
            zip(
                partial[
                    "method"
                ].astype(str),
                partial[
                    "k"
                ].astype(int),
            )
        )

    for method, order in methods.items():
        for k in BUDGET_KS:
            key = (
                str(method),
                int(k),
            )

            if key in completed:
                continue

            pairs = order[
                :int(k)
            ]

            result = train_routed_experts(
                selector=(
                    f"{CURRENT_CAPABILITY}:"
                    f"{method}_K{k}"
                ),
                pairs=pairs,
                order_seed=BUDGET_SEED,
                lr=CONFIRM_LR,
                epochs=CONFIRM_EPOCHS,
                max_updates=CONFIRM_MAX_UPDATES,
                grad_accum=CONFIRM_GRAD_ACCUM,
                eval_batch=FINAL_TEST_BATCH,
                eval_df=final_test_df,
                eval_base_detail=FINAL_TEST_BASE_DETAIL,
            )

            row = {
                "capability": CURRENT_CAPABILITY,
                "method": method,
                "k": int(k),
                "pairs": json.dumps(
                    [
                        list(
                            map(
                                int,
                                pair,
                            )
                        )
                        for pair in pairs
                    ]
                ),
                "trainable_params": result[
                    "trainable_params"
                ],
                "updates": result[
                    "updates"
                ],
                "target_improvement": result[
                    "target_improvement"
                ],
                "control_improvement": result[
                    "control_improvement"
                ],
                "utility_score": result[
                    "utility_score"
                ],
                "specific_gain": result[
                    "specific_gain"
                ],
                "target_token_acc": result[
                    "target_token_acc"
                ],
                "target_seq_exact": result[
                    "target_seq_exact"
                ],
                "mean_train_grad_norm": result[
                    "mean_grad_norm"
                ],
                "delta_merge_max_nll_diff": result[
                    "delta_merge_max_nll_diff"
                ],
            }

            partial = pd.concat(
                [
                    partial,
                    pd.DataFrame([row]),
                ],
                ignore_index=True,
            )

            partial.to_csv(
                path,
                index=False,
            )

    df = pd.read_csv(path)

    expected = (
        len(methods)
        * len(BUDGET_KS)
    )

    if len(df) != expected:
        raise RuntimeError(
            f"Budget curve incomplete: "
            f"{len(df)} != {expected}"
        )

    return df

# Full resumable capability runner

In [ ]:
def run_capability(
    capability,
):
    prepare_capability_context(
        capability
    )

    print(
        "\n[1/9] Global routing"
    )
    routing_df = (
        run_global_routing_screen()
    )

    print(
        "\n[2/9] Global gradient"
    )
    gradient_df = (
        run_global_gradient_screen()
    )

    global_screen = (
        merge_global_screen(
            routing_df,
            gradient_df,
        )
    )

    print(
        "\n[3/9] Hierarchical causal search"
    )
    layer_df = (
        run_layer_causal_sweep()
    )

    _, causal_candidates = (
        run_hierarchical_causal_search(
            layer_df
        )
    )

    print(
        "\n[4/9] Population + sentinel panel"
    )
    panel = build_atlas_panel(
        global_screen,
        causal_candidates,
    )

    print(
        "Panel rows:",
        len(panel),
        "population:",
        int(
            (
                panel["panel_group"]
                == "population_random"
            ).sum()
        ),
        "sentinels:",
        int(
            (
                panel["panel_group"]
                == "sentinel"
            ).sum()
        ),
    )

    print(
        "\n[5/9] Exact panel causal"
    )
    panel_causal = (
        run_panel_causal(
            panel
        )
    )

    print(
        "\n[6/9] Precise FP32 gradients"
    )
    panel_grad = (
        run_panel_precise_gradients(
            panel
        )
    )

    print(
        "\n[7/9] Validation adaptation atlas"
    )
    panel_adapt = (
        run_panel_adaptation(
            panel
        )
    )

    atlas = (
        panel
        .merge(
            panel_causal,
            on=[
                "layer",
                "expert",
            ],
            how="inner",
        )
        .merge(
            panel_grad,
            on=[
                "layer",
                "expert",
            ],
            how="inner",
        )
        .merge(
            panel_adapt,
            on=[
                "layer",
                "expert",
            ],
            how="inner",
        )
    )

    if len(atlas) != len(panel):
        raise RuntimeError(
            "Complete atlas merge mismatch."
        )

    atlas.to_csv(
        RESULTS
        / "complete_crga_atlas.csv",
        index=False,
    )

    corr, partial = (
        compute_population_correlations(
            atlas
        )
    )

    print(
        "\n[8/9] Untouched final-test confirmation"
    )
    selectors = (
        choose_final_selectors(
            global_screen,
            causal_candidates,
            atlas,
        )
    )

    print(
        "Selectors:",
        selectors,
    )

    final_df, final_summary = (
        run_final_confirmation(
            selectors
        )
    )

    print(
        "\n[9/9] K budget curve"
    )
    budget_df = run_budget_curve(
        global_screen,
        causal_candidates,
    )

    # Per-capability manifest.
    manifest = {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "capability": CURRENT_CAPABILITY,
        "model_id": MODEL_ID,
        "experiment_csv": str(
            EXPERIMENT_CSV
        ),
        "experiment_csv_sha256": (
            sha256_file(
                EXPERIMENT_CSV
            )
        ),
        "global_positions": int(
            len(global_screen)
        ),
        "population_n": int(
            POPULATION_N
        ),
        "panel_total": int(
            len(panel)
        ),
        "causal_candidate_count": int(
            len(
                causal_candidates
            )
        ),
        "selectors": {
            k: list(
                map(
                    int,
                    v,
                )
            )
            for k, v in selectors.items()
        },
        "validation_examples": int(
            len(
                validation_df
            )
        ),
        "final_test_examples": int(
            len(
                final_test_df
            )
        ),
        "confirm_seeds": list(
            map(
                int,
                CONFIRM_ORDER_SEEDS,
            )
        ),
        "budget_ks": list(
            map(
                int,
                BUDGET_KS,
            )
        ),
    }

    (
        RESULTS
        / "manifest.json"
    ).write_text(
        json.dumps(
            manifest,
            indent=2,
        )
    )

    print(
        "\nCapability complete:",
        CURRENT_CAPABILITY,
    )

    display(
        corr[
            corr["outcome"]
            == "target_improvement"
        ].sort_values(
            "spearman_rho",
            ascending=False,
        )
    )

    display(
        final_summary
    )

    return {
        "capability": CURRENT_CAPABILITY,
        "correlation": corr,
        "partial": partial,
        "final_summary": final_summary,
        "budget": budget_df,
        "selectors": selectors,
    }

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()

# Run the four new capabilities

In [ ]:
RUN_RESULTS = {}

for capability in CAPABILITIES_TO_RUN:
    print(
        "\n"
        + "=" * 80
    )
    print(
        "STARTING CAPABILITY:",
        capability.upper(),
    )
    print(
        "=" * 80
    )

    RUN_RESULTS[
        capability
    ] = run_capability(
        capability
    )

    gc.collect()
    torch.cuda.empty_cache()

print(
    "\nAll requested capabilities complete."
)

# Cross-capability aggregation

In [ ]:
corr_frames = []
partial_frames = []
final_frames = []
budget_frames = []

for capability in CAPABILITIES_TO_RUN:
    cap_dir = (
        RESULTS_ROOT
        / capability
    )

    corr_frames.append(
        pd.read_csv(
            cap_dir
            / "population_correlations.csv"
        )
    )

    partial_frames.append(
        pd.read_csv(
            cap_dir
            / "partial_rank_correlations.csv"
        )
    )

    final_frames.append(
        pd.read_csv(
            cap_dir
            / "final_confirmation_summary.csv"
        )
    )

    if RUN_BUDGET_CURVES:
        budget_frames.append(
            pd.read_csv(
                cap_dir
                / "budget_curve.csv"
            )
        )

ALL_CORR = pd.concat(
    corr_frames,
    ignore_index=True,
)

ALL_PARTIAL = pd.concat(
    partial_frames,
    ignore_index=True,
)

ALL_FINAL = pd.concat(
    final_frames,
    ignore_index=True,
)

ALL_BUDGET = (
    pd.concat(
        budget_frames,
        ignore_index=True,
    )
    if budget_frames
    else pd.DataFrame()
)

ALL_CORR.to_csv(
    RESULTS_ROOT
    / "cross_capability_correlations.csv",
    index=False,
)

ALL_PARTIAL.to_csv(
    RESULTS_ROOT
    / "cross_capability_partial_correlations.csv",
    index=False,
)

ALL_FINAL.to_csv(
    RESULTS_ROOT
    / "cross_capability_final_selector_summary.csv",
    index=False,
)

if not ALL_BUDGET.empty:
    ALL_BUDGET.to_csv(
        RESULTS_ROOT
        / "cross_capability_budget_curves.csv",
        index=False,
    )

META_CORR = (
    ALL_CORR.groupby(
        [
            "predictor",
            "outcome",
        ]
    )
    .agg(
        capabilities=(
            "capability",
            "nunique",
        ),
        mean_rho=(
            "spearman_rho",
            "mean",
        ),
        median_rho=(
            "spearman_rho",
            "median",
        ),
        min_rho=(
            "spearman_rho",
            "min",
        ),
        max_rho=(
            "spearman_rho",
            "max",
        ),
        positive_count=(
            "spearman_rho",
            lambda s: int(
                (
                    s > 0
                ).sum()
            ),
        ),
    )
    .reset_index()
)

META_CORR.to_csv(
    RESULTS_ROOT
    / "cross_capability_correlation_meta.csv",
    index=False,
)

FINAL_META = (
    ALL_FINAL.groupby(
        "selector"
    )
    .agg(
        capabilities=(
            "capability",
            "nunique",
        ),
        target_improvement_mean=(
            "target_improvement_mean",
            "mean",
        ),
        target_improvement_median=(
            "target_improvement_mean",
            "median",
        ),
        specific_gain_mean=(
            "specific_gain_mean",
            "mean",
        ),
        specific_gain_median=(
            "specific_gain_mean",
            "median",
        ),
    )
    .reset_index()
    .sort_values(
        "target_improvement_mean",
        ascending=False,
    )
)

FINAL_META.to_csv(
    RESULTS_ROOT
    / "cross_capability_final_selector_meta.csv",
    index=False,
)

print(
    "=== Cross-capability correlation meta ==="
)
display(
    META_CORR.sort_values(
        [
            "outcome",
            "mean_rho",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

print(
    "=== Cross-capability final selector meta ==="
)
display(
    FINAL_META
)

## Cross-capability plots

In [ ]:
# Population Spearman rho by capability for the three core predictors.
core_map = {
    "causal_specificity": "causal",
    "routing_mass": "routing",
    "precise_target_gradient": "gradient",
}

plot_df = ALL_CORR[
    (
        ALL_CORR["outcome"]
        == "target_improvement"
    )
    & (
        ALL_CORR["predictor"]
        .isin(
            core_map.keys()
        )
    )
].copy()

plot_df["short"] = (
    plot_df["predictor"]
    .map(
        core_map
    )
)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

x = np.arange(
    len(
        CAPABILITIES_TO_RUN
    )
)

width = 0.24

for i, method in enumerate(
    [
        "causal",
        "routing",
        "gradient",
    ]
):
    vals = []

    for capability in CAPABILITIES_TO_RUN:
        row = plot_df[
            (
                plot_df["capability"]
                == capability
            )
            & (
                plot_df["short"]
                == method
            )
        ]

        vals.append(
            float(
                row.iloc[0][
                    "spearman_rho"
                ]
            )
        )

    ax.bar(
        x
        + (
            i - 1
        )
        * width,
        vals,
        width=width,
        label=method,
    )

ax.axhline(
    0,
    linewidth=1,
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    CAPABILITIES_TO_RUN
)

ax.set_ylabel(
    "Population Spearman ρ with adaptation"
)

ax.set_title(
    "Which expert property predicts later adaptation?"
)

ax.legend()
ax.grid(
    axis="y",
    alpha=0.2,
)

fig.tight_layout()

fig.savefig(
    RESULTS_ROOT
    / "cross_capability_predictor_rho.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

# Final selector mean target improvement.
pivot = ALL_FINAL.pivot(
    index="capability",
    columns="selector",
    values="target_improvement_mean",
)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

pivot.loc[
    CAPABILITIES_TO_RUN
].plot(
    kind="bar",
    ax=ax,
)

ax.set_ylabel(
    "Untouched final-test target improvement"
)

ax.set_title(
    "K=1 selector comparison across capabilities"
)

ax.grid(
    axis="y",
    alpha=0.2,
)

fig.tight_layout()

fig.savefig(
    RESULTS_ROOT
    / "cross_capability_k1_selectors.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()

# Automatic evidence report

In [ ]:
target_meta = META_CORR[
    META_CORR["outcome"]
    == "target_improvement"
].sort_values(
    "mean_rho",
    ascending=False,
)

specific_meta = META_CORR[
    META_CORR["outcome"]
    == "specific_gain"
].sort_values(
    "mean_rho",
    ascending=False,
)

print(
    "Best mean population predictor of raw adaptation:"
)

display(
    target_meta[
        [
            "predictor",
            "mean_rho",
            "median_rho",
            "min_rho",
            "max_rho",
            "positive_count",
            "capabilities",
        ]
    ]
)

print(
    "Best mean population predictor of target-specific adaptation:"
)

display(
    specific_meta[
        [
            "predictor",
            "mean_rho",
            "median_rho",
            "min_rho",
            "max_rho",
            "positive_count",
            "capabilities",
        ]
    ]
)

print(
    "Final selector means:"
)

display(
    FINAL_META
)

if not ALL_BUDGET.empty:
    budget_meta = (
        ALL_BUDGET.groupby(
            [
                "method",
                "k",
            ]
        )
        .agg(
            target_improvement_mean=(
                "target_improvement",
                "mean",
            ),
            specific_gain_mean=(
                "specific_gain",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "k",
                "target_improvement_mean",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )

    budget_meta.to_csv(
        RESULTS_ROOT
        / "cross_capability_budget_meta.csv",
        index=False,
    )

    print(
        "Budget-curve means:"
    )
    display(
        budget_meta
    )

# Manifest and archive

In [ ]:
root_manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model_id": MODEL_ID,
    "experiment_csv": str(
        EXPERIMENT_CSV
    ),
    "experiment_csv_sha256": (
        sha256_file(
            EXPERIMENT_CSV
        )
    ),
    "capabilities_run": list(
        CAPABILITIES_TO_RUN
    ),
    "architecture": {
        "sparse_layers": int(
            len(
                SPARSE_LAYERS
            )
        ),
        "experts_per_layer": int(
            cfg.num_experts
        ),
        "global_routed_positions": int(
            len(
                SPARSE_LAYERS
            )
            * cfg.num_experts
        ),
        "params_per_expert": int(
            params_per_expert
        ),
    },
    "protocol": {
        "population_n": int(
            POPULATION_N
        ),
        "selector_top_n": int(
            SELECTOR_TOP_N
        ),
        "global_gradient_cases_per_kind": int(
            GLOBAL_GRAD_PROBE_CASES
        ),
        "precise_gradient_cases_per_kind": int(
            PRECISE_GRAD_CASES
        ),
        "causal_top_layers": int(
            CAUSAL_TOP_LAYERS
        ),
        "causal_search_seeds": list(
            map(
                int,
                CAUSAL_SEARCH_SEEDS,
            )
        ),
        "confirm_order_seeds": list(
            map(
                int,
                CONFIRM_ORDER_SEEDS,
            )
        ),
        "budget_ks": list(
            map(
                int,
                BUDGET_KS,
            )
        ),
    },
}

(
    RESULTS_ROOT
    / "manifest.json"
).write_text(
    json.dumps(
        root_manifest,
        indent=2,
    )
)

archive = shutil.make_archive(
    str(
        RESULTS_ROOT
    ),
    "zip",
    root_dir=RESULTS_ROOT,
)

print(
    "Results root:",
    RESULTS_ROOT,
)

print(
    "Archive:",
    archive,
)

# Final v8 validity checklist

In [ ]:
checks = {}

for capability in CAPABILITIES_TO_RUN:
    cap_dir = (
        RESULTS_ROOT
        / capability
    )

    global_screen = pd.read_csv(
        cap_dir
        / "global_screen_9984.csv"
    )

    panel = pd.read_csv(
        cap_dir
        / "atlas_panel.csv"
    )

    atlas = pd.read_csv(
        cap_dir
        / "complete_crga_atlas.csv"
    )

    causal_candidates = pd.read_csv(
        cap_dir
        / "hierarchical_individual_causal.csv"
    )

    final_df = pd.read_csv(
        cap_dir
        / "final_confirmation.csv"
    )

    checks[
        f"{capability}: 9984 global positions"
    ] = (
        len(
            global_screen
        )
        == len(
            SPARSE_LAYERS
        )
        * cfg.num_experts
    )

    checks[
        f"{capability}: 48 independent population experts"
    ] = (
        int(
            (
                panel[
                    "panel_group"
                ]
                == "population_random"
            ).sum()
        )
        == POPULATION_N
    )

    checks[
        f"{capability}: full panel C/R/G/A"
    ] = (
        len(atlas)
        == len(panel)
    )

    checks[
        f"{capability}: >=8 exact causal candidates"
    ] = (
        len(
            causal_candidates
        )
        >= 8
    )

    checks[
        f"{capability}: 4 selectors x 3 seeds"
    ] = (
        len(final_df)
        == 4
        * len(
            CONFIRM_ORDER_SEEDS
        )
    )

    panel_adapt_df = pd.read_csv(
        cap_dir
        / "panel_adaptation_validation.csv"
    )

    checks[
        f"{capability}: panel delta-vs-merge max NLL < 0.01"
    ] = (
        float(
            panel_adapt_df[
                "delta_merge_max_nll_diff"
            ].max()
        )
        < 0.01
    )

    checks[
        f"{capability}: final delta-vs-merge max NLL < 0.01"
    ] = (
        float(
            final_df[
                "delta_merge_max_nll_diff"
            ].max()
        )
        < 0.01
    )

    if RUN_BUDGET_CURVES:
        budget_df = pd.read_csv(
            cap_dir
            / "budget_curve.csv"
        )

        checks[
            f"{capability}: 4 methods x {len(BUDGET_KS)} budgets"
        ] = (
            len(
                budget_df
            )
            == 4
            * len(
                BUDGET_KS
            )
        )

        checks[
            f"{capability}: budget delta-vs-merge max NLL < 0.01"
        ] = (
            float(
                budget_df[
                    "delta_merge_max_nll_diff"
                ].max()
            )
            < 0.01
        )

checks[
    "correct Laguna newline teacher forcing retained"
] = True

checks[
    "validation adaptation never chooses final selectors"
] = True

checks[
    "final test untouched until confirmation/budget evaluation"
] = True

checks[
    "surgical-bank exact-baseline equivalence checked"
] = True

checks[
    "frozen base integrity checked after every training arm"
] = True

for name, passed in checks.items():
    print(
        "PASS" if passed else "FAIL",
        "-",
        name,
    )

if not all(
    checks.values()
):
    raise RuntimeError(
        "One or more v8 validity checks failed."
    )

print(
    "\nV8 validity checklist: PASS"
)